# EXP-01 — Baseline ResNet50 224px (CBIS-DDSM)
## Execução em Kaggle com tracking no MLflow

Notebook equivalente ao comando local/Docker (`train.py --config config_exp01.yaml`
+ `predict.py`), com rastreamento de experimentos no MLflow seguindo o
**Padrão de tracking — MLflow** (tags, parâmetros, métricas e artefatos
obrigatórios).

**Dependências:** S3-01, S3-02 e configuração mínima do MLflow (servidor já
provisionado — este notebook só se conecta a ele via `MLFLOW_TRACKING_URI`,
não configura o servidor).

**Fluxo deste notebook:**
1. Checklist de entrada (dataset/manifest, split, seed, config, ambiente)
2. Conexão ao MLflow (credenciais via Kaggle Secrets, nunca hardcoded)
3. **Smoke test** — obrigatório rodar e passar antes do run completo
4. Run completo — treino + predição no teste congelado + métricas com IC 95%

**Segurança de dados (crítico):** nenhum arquivo DICOM, imagem clínica,
caminho pessoal de máquina ou credencial é publicado neste notebook ou no
seu output. Checkpoints e CSVs de predição vão *apenas* para o MLflow
(artifact store próprio, não o output público do Kaggle) e são apagados da
pasta de trabalho imediatamente após o upload.

## Checklist de entrada (leitura manual antes de rodar)

Confirme cada item **antes** de rodar a célula do smoke test. A célula de
código logo abaixo também valida boa parte disso automaticamente, mas a
leitura manual continua sendo a checagem principal.

| Item | Confirmação |
|---|---|
| **Dataset / manifest** | O dataset CBIS-DDSM está anexado ao notebook (`/kaggle/input/...`) e os 4 CSVs oficiais (mass/calc × train/test) existem no caminho configurado |
| **Split** | `train_csvs` e `test_csvs` apontam para arquivos diferentes — o teste é o split oficial do CBIS-DDSM, nunca re-sorteado |
| **Seed** | `CONFIG["seed"]` está definida e é a mesma usada em runs anteriores comparáveis |
| **Configuração** | `CONFIG` revisado: arquitetura, resolução, batch, LR, épocas correspondem ao experimento pretendido (ex: EXP-01 na matriz de experimentos) |
| **Ambiente** | Rodando em Kaggle (ou nuvem aprovada) com GPU do tipo esperado — confira em Settings > Accelerator |
| **Segredos MLflow** | `MLFLOW_TRACKING_URI` (e token, se aplicável) cadastrados em Add-ons > Secrets — nunca colados no código |
| **Orçamento de GPU** | Custo estimado deste run já checado contra o limite semanal (ver planilha de plano de execução) antes de rodar o run completo |


In [11]:
import os

base = "/kaggle/input/datasets"
for root, dirs, files in os.walk(base):
    depth = root.replace(base, '').count(os.sep)
    if depth > 3:
        dirs[:] = []
        continue
    print(root)
    for f in files[:5]:
        print("   ", f)

/kaggle/input/datasets
/kaggle/input/datasets/awsaf49
/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset
/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset/jpeg
/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset/csv
    dicom_info.csv
    mass_case_description_train_set.csv
    calc_case_description_train_set.csv
    meta.csv
    calc_case_description_test_set.csv
/kaggle/input/datasets/joovitorbach
/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado-s4
    Dockerfile
    evaluate.py
    README.md
    requirements.txt
/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado-s4/tests
    test_segmentation_metrics.py
    test_classification_metrics.py
    test_evaluate_cli.py
    test_io_validation.py
/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado-s4/evaluate
    metrics_segmentation.py
    metrics_classification.py
    io_utils.py
    __init__.py
/kaggle/input/datasets/joovitorbach/mmlflow1
    mlflow(1)

In [13]:
import os
os.environ['CODE_VERSION'] = '9572f3c'

import os, sys, subprocess, json, hashlib, time
from datetime import datetime

# ============================================================
# CONFIG — edite apenas estes valores antes de rodar.
# NUNCA coloque caminho pessoal de máquina local aqui (ex: C:\Users\...).
# Neste notebook, os dados vêm sempre de /kaggle/input/... (dataset Kaggle).
# ============================================================
CONFIG = {
    "experiment_id": "EXP-01",
    "owner": "Guilherme BettegaS",          # <-- troque pelo seu usuário/identificador de papel
    "role": "02",                     # <-- número do papel responsável (ver padrão de tags)
    "ablation_axis": "resolution",    # eixo variado nesta rodada (baseline = ponto de partida do eixo "resolution")
    "seed": 42,
    "architecture": "resnet50",
    "pretrained_weights": "imagenet",
    "image_size": 224,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "epochs": 20,
    "dataset_root": "/kaggle/input/datasets/awsaf49/cbis-ddsm-breast-cancer-image-dataset",
    "train_csvs": [
        "csv/mass_case_description_train_set.csv",
        "csv/calc_case_description_train_set.csv",
    ],
    "test_csvs": [
        "csv/mass_case_description_test_set.csv",
        "csv/calc_case_description_test_set.csv",
    ],
    "image_path_column": "cropped image file path",
    "patient_id_column": "patient_id",
    "pathology_column": "pathology",
    "malignant_labels": ["MALIGNANT"],
    "non_malignant_labels": ["BENIGN", "BENIGN_WITHOUT_CALLBACK"],
    "val_fraction": 0.15,
    "preprocessing_version": "preprocessing@REPLACE_WITH_COMMIT_OR_TAG",
}

WORKDIR = "/kaggle/working"
os.makedirs(WORKDIR, exist_ok=True)


def get_code_version():
    """Hash do commit Git ativo. Em notebooks Kaggle não há .git local por
    padrão — se este notebook foi gerado a partir de um script versionado,
    defina a variável de ambiente CODE_VERSION antes de rodar (ex: via
    Kaggle Secrets ou como parâmetro do kernel). Caso contrário, cai para
    um placeholder explícito (NUNCA deixe isso como 'unknown' num run real)."""
    env_version = os.environ.get("CODE_VERSION")
    if env_version:
        return env_version
    try:
        out = subprocess.check_output(["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL)
        return out.decode().strip()
    except Exception:
        return "UNKNOWN_SET_CODE_VERSION_ENV_VAR"


def detect_environment():
    """Detecta o ambiente de execução para a tag `environment`."""
    if os.path.exists("/kaggle"):
        try:
            gpu_info = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                stderr=subprocess.DEVNULL,
            ).decode().strip()
            return "kaggle-gpu" if gpu_info else "kaggle-cpu"
        except Exception:
            return "kaggle-cpu"
    return "local-cpu"


CODE_VERSION = get_code_version()
ENVIRONMENT = detect_environment()

print(f"CODE_VERSION: {CODE_VERSION}")
print(f"ENVIRONMENT detectado: {ENVIRONMENT}")


CODE_VERSION: 9572f3c
ENVIRONMENT detectado: kaggle-gpu


### Checklist de entrada — validação automática

In [15]:
# ============================================================
# CHECKLIST DE ENTRADA — roda automaticamente, mas NÃO substitui a leitura
# manual da checklist.md antes de aprovar a execução completa.
# Falha (assert) se algo obrigatório estiver faltando, em vez de deixar o
# treino começar com uma configuração incompleta.
# ============================================================

checklist_status = {}

# 1. Dataset / manifest presente
dataset_ok = os.path.isdir(CONFIG["dataset_root"])
checklist_status["dataset_root_existe"] = dataset_ok

manifest_files_ok = []
for csv_list_name in ["train_csvs", "test_csvs"]:
    for rel_path in CONFIG[csv_list_name]:
        full_path = os.path.join(CONFIG["dataset_root"], rel_path)
        manifest_files_ok.append(os.path.exists(full_path))
checklist_status["todos_os_csvs_encontrados"] = all(manifest_files_ok)

# 2. Split — confirma que vamos usar o split OFICIAL (arquivos de teste
# separados dos de treino, nunca a mesma lista)
checklist_status["split_train_test_distintos"] = (
    set(CONFIG["train_csvs"]).isdisjoint(set(CONFIG["test_csvs"]))
)

# 3. Seed definida
checklist_status["seed_definida"] = isinstance(CONFIG["seed"], int)

# 4. Configuração sem caminho pessoal de máquina local
config_str = json.dumps(CONFIG)
personal_path_patterns = ["C:\\Users", "/home/", "/Users/"]
checklist_status["sem_caminho_pessoal_no_config"] = not any(
    p in config_str for p in personal_path_patterns
)

# 5. Ambiente detectado é um ambiente aprovado (Kaggle ou nuvem aprovada)
checklist_status["ambiente_aprovado"] = ENVIRONMENT.startswith("kaggle") or ENVIRONMENT == "docker"

# 6. code_version não é o placeholder de erro
checklist_status["code_version_definida"] = CODE_VERSION != "UNKNOWN_SET_CODE_VERSION_ENV_VAR"

print("=== CHECKLIST DE ENTRADA ===")
for item, ok in checklist_status.items():
    print(f"  [{'OK' if ok else 'FALTA'}] {item}")

obrigatorios = ["dataset_root_existe", "todos_os_csvs_encontrados",
                 "split_train_test_distintos", "seed_definida",
                 "sem_caminho_pessoal_no_config", "ambiente_aprovado"]
faltando = [k for k in obrigatorios if not checklist_status[k]]

if faltando:
    raise RuntimeError(
        f"Checklist de entrada incompleto, itens faltando: {faltando}. "
        "Corrija antes de rodar o smoke test ou o treino completo."
    )

if not checklist_status["code_version_definida"]:
    print("\n[AVISO] CODE_VERSION não definida via variável de ambiente. "
          "Defina antes do run completo (não bloqueante para o smoke test).")

print("\nChecklist obrigatório: OK. Pode prosseguir para o smoke test.")


=== CHECKLIST DE ENTRADA ===
  [OK] dataset_root_existe
  [OK] todos_os_csvs_encontrados
  [OK] split_train_test_distintos
  [OK] seed_definida
  [OK] sem_caminho_pessoal_no_config
  [OK] ambiente_aprovado
  [OK] code_version_definida

Checklist obrigatório: OK. Pode prosseguir para o smoke test.


### Dataset — carregamento dos CSVs oficiais e split por paciente

In [36]:
# dataset.py
"""Dataset do CBIS-DDSM para classificação binária maligno x não maligno em
patches (imagens já recortadas na ROI da lesão).

CORREÇÃO IMPORTANTE (descoberta ao rodar no Kaggle): a coluna
'cropped image file path' dos CSVs mass/calc_case_description_*.csv aponta
para o caminho do arquivo DICOM ORIGINAL (.dcm), que não existe mais nesse
download — só as versões .jpg convertidas existem, dentro de jpeg/<SeriesInstanceUID>/.
Para achar o .jpg certo, usamos o csv/dicom_info.csv como tabela de tradução:
ele tem uma linha por imagem, com 'SeriesInstanceUID' e 'image_path' (o
caminho relativo real do .jpg, tipo 'CBIS-DDSM/jpeg/<UID>/1-172.jpg').

A tradução funciona assim:
    1. Extrai o SeriesInstanceUID do caminho bruto (.dcm) do CSV de casos
       — é o penúltimo segmento do caminho, logo antes do nome do arquivo.
    2. Procura esse SeriesInstanceUID no dicom_info.csv.
    3. Usa o 'image_path' correspondente (já aponta pro .jpg real).
"""

import os
import re
from typing import List, Optional

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms


# Um UID de DICOM é uma sequência de números separados por ponto (ex:
# "1.3.6.1.4.1.9590.100.1.2.128573...").
_UID_PATTERN = re.compile(r"^\d+(\.\d+)+$")


def load_and_merge_csvs(dataset_root: str, csv_paths: List[str]) -> pd.DataFrame:
    dfs = []
    for rel_path in csv_paths:
        full_path = os.path.join(dataset_root, rel_path)
        if not os.path.exists(full_path):
            raise FileNotFoundError(
                f"CSV não encontrado: {full_path}\n"
                "Confira 'dataset_root' e 'train_csvs'/'test_csvs' no config."
            )
        df = pd.read_csv(full_path)
        df.columns = [c.strip() for c in df.columns]
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)


def label_from_pathology(pathology: str, malignant_labels, non_malignant_labels) -> Optional[int]:
    pathology = str(pathology).strip().upper()
    if pathology in [p.upper() for p in malignant_labels]:
        return 1
    if pathology in [p.upper() for p in non_malignant_labels]:
        return 0
    return None


def extract_series_uid(raw_path: str) -> Optional[str]:
    """Extrai o SeriesInstanceUID de um caminho bruto do CSV de casos.

    Exemplo de entrada:
      '.../Mass-Training_P_00004_RIGHT_MLO_1/1.3.6.1...774/1.3.6.1...846/000000.dcm'
    O SeriesInstanceUID é o segmento de UID imediatamente ANTES do nome do
    arquivo (o mais profundo dos dois UIDs no caminho).
    """
    raw_path = str(raw_path).replace("\\", "/").strip()
    parts = [p for p in raw_path.split("/") if p]
    if len(parts) < 2:
        return None
    # O penúltimo segmento (logo antes do nome do arquivo .dcm) é o SeriesInstanceUID
    candidate = parts[-2]
    if _UID_PATTERN.match(candidate):
        return candidate
    # fallback: procura o último segmento que pareça um UID, de trás pra frente
    for p in reversed(parts[:-1]):
        if _UID_PATTERN.match(p):
            return p
    return None


def build_dicom_info_lookup(dataset_root: str, dicom_info_rel_path: str = "csv/dicom_info.csv") -> dict:
    """SeriesInstanceUID -> {"crop": ..., "mask": ..., "full": ..., "single": ...}
    (caminhos relativos a dataset_root). Em séries com 2 arquivos, o papel é
    decidido pela SeriesDescription quando as DUAS linhas a trazem; caso
    contrário, pelo tamanho: a máscara tem o tamanho da mamografia inteira e
    o recorte é sempre menor."""
    full_path = os.path.join(dataset_root, dicom_info_rel_path)
    if not os.path.exists(full_path):
        raise FileNotFoundError(f"dicom_info.csv não encontrado em {full_path}.")
    df = pd.read_csv(full_path)
    df.columns = [c.strip() for c in df.columns]
 
    for col in ["SeriesInstanceUID", "image_path", "SeriesDescription", "Rows", "Columns"]:
        if col not in df.columns:
            raise KeyError(f"dicom_info.csv sem a coluna '{col}'. Colunas: {list(df.columns)}")
 
    df["_uid"] = df["SeriesInstanceUID"].astype(str).str.strip()
    df["_path"] = df["image_path"].astype(str).str.strip().str.replace("\\", "/", regex=False)
    df["_path"] = df["_path"].apply(lambda p: p[len("CBIS-DDSM/"):] if p.startswith("CBIS-DDSM/") else p)
    df["_area"] = pd.to_numeric(df["Rows"], errors="coerce") * pd.to_numeric(df["Columns"], errors="coerce")
    desc_to_role = {"cropped images": "crop", "ROI mask images": "mask", "full mammogram images": "full"}
    df["_role"] = df["SeriesDescription"].astype(str).str.strip().map(desc_to_role)
 
    lookup, n_pairs, n_conflicts = {}, 0, 0
    for uid, g in df.groupby("_uid", sort=False):
        entry = {}
        if len(g) == 1:
            r = g.iloc[0]
            entry["single"] = r["_path"]
            if isinstance(r["_role"], str):
                entry[r["_role"]] = r["_path"]
        else:
            n_pairs += 1
            g = g.sort_values("_area", na_position="last")
            by_area = {"crop": g.iloc[0], "mask": g.iloc[-1]}  # menor = recorte, maior = máscara
            if set(g["_role"]) == {"crop", "mask"}:            # descrição completa nas duas linhas
                chosen = {r["_role"]: r for _, r in g.iterrows()}
                if chosen["crop"]["_path"] != by_area["crop"]["_path"]:
                    n_conflicts += 1
            else:
                chosen = by_area
            entry["crop"], entry["mask"] = chosen["crop"]["_path"], chosen["mask"]["_path"]
        lookup[uid] = entry
 
    print(f"[lookup] {len(lookup)} séries; {n_pairs} com par recorte/máscara; "
          f"{n_conflicts} em que a descrição contradiz o tamanho.")
    return lookup


def resolve_image_path(dataset_root: str, raw_path: str, dicom_lookup: dict, role: str = "crop") -> str:
    """Traduz o caminho bruto do CSV (.dcm) para o .jpg real. role: "crop"
    (padrão, usado pelo classificador), "mask" (máscara ROI) ou "full"
    (mamografia inteira)."""
    series_uid = extract_series_uid(raw_path)
 
    if series_uid and series_uid in dicom_lookup:
        entry = dicom_lookup[series_uid]
        rel = entry.get(role) or entry.get("single")
        if rel:
            return os.path.join(dataset_root, rel)
 
    raw_path_clean = str(raw_path).replace("\\", "/").strip()
    if os.path.isabs(raw_path_clean) and os.path.exists(raw_path_clean):
        return raw_path_clean
 
    marker = "jpeg/"
    idx = raw_path_clean.find(marker)
    if idx != -1:
        candidate = os.path.join(dataset_root, raw_path_clean[idx:])
        if os.path.exists(candidate):
            return candidate
 
    return os.path.join(dataset_root, raw_path_clean.lstrip("/"))


class CBISDDSMPatchDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, dataset_root: str, image_path_column: str,
                 resolution: int, normalize_mean, normalize_std, augment: bool = False,
                 dicom_lookup: Optional[dict] = None):
        self.df = dataframe.reset_index(drop=True)
        self.dataset_root = dataset_root
        self.image_path_column = image_path_column
        self.dicom_lookup = dicom_lookup or {}

        if augment:
            self.transform = transforms.Compose([
                transforms.Resize((resolution, resolution)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.2),
                transforms.RandomRotation(degrees=10),
                transforms.ToTensor(),
                transforms.Normalize(mean=normalize_mean, std=normalize_std),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((resolution, resolution)),
                transforms.ToTensor(),
                transforms.Normalize(mean=normalize_mean, std=normalize_std),
            ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = resolve_image_path(self.dataset_root, row[self.image_path_column], self.dicom_lookup)

        try:
            img = Image.open(img_path).convert("RGB")
        except Exception as e:
            raise RuntimeError(
                f"Falha ao abrir imagem '{img_path}' (linha {idx} do dataframe, "
                f"paciente {row.get('patient_id', '?')}). Caminho original no CSV: "
                f"'{row[self.image_path_column]}'. Erro original: {e}"
            )

        img_t = self.transform(img)
        label = torch.tensor(float(row["label"]), dtype=torch.float32)

        return img_t, label, str(row.get("patient_id", "")), img_path


def build_labeled_dataframe(dataset_root: str, csv_paths: List[str], malignant_labels,
                             non_malignant_labels, patient_id_column: str,
                             pathology_column: str) -> pd.DataFrame:
    df = load_and_merge_csvs(dataset_root, csv_paths)

    if patient_id_column not in df.columns:
        raise KeyError(
            f"Coluna '{patient_id_column}' não encontrada no CSV. "
            f"Colunas disponíveis: {list(df.columns)}"
        )
    if pathology_column not in df.columns:
        raise KeyError(
            f"Coluna '{pathology_column}' não encontrada no CSV. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    df["label"] = df[pathology_column].apply(
        lambda p: label_from_pathology(p, malignant_labels, non_malignant_labels)
    )
    n_before = len(df)
    df = df.dropna(subset=["label"]).reset_index(drop=True)
    n_after = len(df)
    if n_after < n_before:
        print(f"[dataset] Descartadas {n_before - n_after} linhas com rótulo de patologia "
              f"desconhecido (fora de {malignant_labels + non_malignant_labels}).")

    df["patient_id"] = df[patient_id_column]
    return df


def filter_resolvable_rows(df: pd.DataFrame, dataset_root: str, image_path_column: str,
                            dicom_lookup: dict) -> pd.DataFrame:
    """Remove linhas cujo arquivo de imagem não existe de fato no disco,
    APÓS a tradução via dicom_info.csv. Isso evita que o treino inteiro
    trave por causa de algumas linhas com dados inconsistentes no CSV
    original (comum em datasets grandes reais) — em vez disso, avisa
    quantas/quais foram descartadas e segue com as válidas."""
    resolved_paths = []
    exists_flags = []

    for raw_path in df[image_path_column]:
        resolved = resolve_image_path(dataset_root, raw_path, dicom_lookup)
        resolved_paths.append(resolved)
        exists_flags.append(os.path.exists(resolved))

    df = df.copy()
    df["_resolved_path"] = resolved_paths
    df["_path_exists"] = exists_flags

    n_before = len(df)
    missing_df = df[~df["_path_exists"]]
    df = df[df["_path_exists"]].reset_index(drop=True)
    n_after = len(df)

    if n_after < n_before:
        n_missing = n_before - n_after
        pct = 100 * n_missing / n_before
        print(f"[dataset] AVISO: {n_missing} de {n_before} linhas ({pct:.1f}%) descartadas "
              f"por não terem arquivo de imagem correspondente no disco "
              f"(inconsistência entre o CSV de casos e o dicom_info.csv/pasta jpeg/).")
        if n_missing <= 10:
            print("[dataset] Pacientes afetados:", missing_df["patient_id"].tolist())
        if pct > 5:
            print(f"[dataset] ATENÇÃO: {pct:.1f}% é uma proporção alta — vale investigar "
                  f"antes de confiar nos resultados (pode ser um problema sistemático, "
                  f"não só ruído aleatório).")

    return df.drop(columns=["_resolved_path", "_path_exists"])


def patient_level_split(df: pd.DataFrame, val_fraction: float, seed: int):
    rng = np.random.RandomState(seed)
    patients = df["patient_id"].unique()
    rng.shuffle(patients)

    n_val = max(1, int(len(patients) * val_fraction))
    val_patients = set(patients[:n_val])

    val_df = df[df["patient_id"].isin(val_patients)].reset_index(drop=True)
    train_df = df[~df["patient_id"].isin(val_patients)].reset_index(drop=True)

    return train_df, val_df


In [37]:
# ------------------------------------------------------------------------
# VALIDAÇÃO -- rode DEPOIS de aplicar o patch e reiniciar o kernel. Se o
# patch funcionou: contaminação ~0 no recorte, máscara maior que o recorte,
# e máscara com o mesmo tamanho da mamografia inteira.
# ------------------------------------------------------------------------
def validar_resolucao(config, dicom_lookup, n_amostra=300):
    from PIL import Image
    root = config["dataset_root"]
    img_col, mask_col, full_col = config["image_path_column"], "ROI mask file path", "image file path"
    for nome, csvs in [("treino", config["train_csvs"]), ("teste", config["test_csvs"])]:
        d = build_labeled_dataframe(root, csvs, config["malignant_labels"], config["non_malignant_labels"],
                                     config["patient_id_column"], config["pathology_column"])
        d = filter_resolvable_rows(d, root, img_col, dicom_lookup)
        grandes = sum(max(Image.open(resolve_image_path(root, r[img_col], dicom_lookup, "crop")).size) > 2000
                      for _, r in d.iterrows())
        print(f"{nome}: {grandes}/{len(d)} recortes com tamanho de mamografia inteira (esperado ~0)")
        if nome == "teste":
            amostra = d.sample(min(n_amostra, len(d)), random_state=0)
            mask_maior = mesmo_shape_full = 0
            tem_full = full_col in d.columns
            for _, r in amostra.iterrows():
                c = Image.open(resolve_image_path(root, r[img_col], dicom_lookup, "crop")).size
                m = Image.open(resolve_image_path(root, r[mask_col], dicom_lookup, "mask")).size
                mask_maior += int(m[0] * m[1] > c[0] * c[1])
                if tem_full:
                    f = Image.open(resolve_image_path(root, r[full_col], dicom_lookup, "full")).size
                    mesmo_shape_full += int(m == f)
            print(f"  máscara maior que o recorte: {mask_maior}/{len(amostra)} (esperado ~todas)")
            if tem_full:
                print(f"  máscara com o MESMO tamanho da mamografia inteira: {mesmo_shape_full}/{len(amostra)}")
            else:
                print(f"  (coluna '{full_col}' não existe no dataframe; colunas: {list(d.columns)})")

### Métricas e reprodutibilidade (seeds, AUROC/AUPRC, bootstrap CI)

In [18]:
# utils.py
"""Utilidades de reprodutibilidade e métricas (AUROC, AUPRC, sensibilidade/
especificidade em ponto de operação fixo, com intervalos de confiança de 95%
por bootstrap), conforme exigido no protocolo experimental do projeto."""

import random

import numpy as np
import torch
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, f1_score,
)


def set_seed(seed: int):
    """Fixa a semente em todas as fontes de aleatoriedade relevantes."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo adicional (pode reduzir performance um pouco, mas é mais
    # importante para reprodutibilidade em um projeto acadêmico):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sensitivity_at_specificity(y_true, y_score, target_specificity=0.90):
    """Retorna (sensibilidade, limiar) no ponto em que a especificidade é
    a mais próxima possível (por cima) do valor alvo (padrão: 90%)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    valid = specificity >= target_specificity
    if not valid.any():
        # nenhum ponto atinge a especificidade alvo -> usa o mais próximo
        idx = np.argmax(specificity)
    else:
        # entre os pontos que atingem a especificidade alvo, pega o de
        # maior sensibilidade (tpr)
        candidate_idx = np.where(valid)[0]
        idx = candidate_idx[np.argmax(tpr[candidate_idx])]
    return float(tpr[idx]), float(thresholds[idx])


def specificity_at_sensitivity(y_true, y_score, target_sensitivity=0.90):
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    specificity = 1 - fpr
    valid = tpr >= target_sensitivity
    if not valid.any():
        idx = np.argmax(tpr)
    else:
        candidate_idx = np.where(valid)[0]
        idx = candidate_idx[np.argmax(specificity[candidate_idx])]
    return float(specificity[idx]), float(thresholds[idx])


def compute_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    y_pred = (y_score >= threshold).astype(int)

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)
    sens90, thr_sens90 = sensitivity_at_specificity(y_true, y_score, 0.90)
    spec90, thr_spec90 = specificity_at_sensitivity(y_true, y_score, 0.90)
    cm = confusion_matrix(y_true, y_pred).tolist()
    f1 = f1_score(y_true, y_pred)

    return {
        "auroc": auroc,
        "auprc": auprc,
        "sensitivity_at_specificity_90": sens90,
        "specificity_at_sensitivity_90": spec90,
        "f1_at_threshold_0.5": f1,
        "confusion_matrix_at_threshold_0.5": cm,
        "n_samples": len(y_true),
        "n_positive": int(y_true.sum()),
    }


def bootstrap_ci(y_true, y_score, metric_fn, n_bootstrap=1000, ci=0.95, seed=42):
    """Intervalo de confiança por bootstrap (reamostragem com reposição),
    conforme exigido no protocolo (mínimo 1000 reamostragens)."""
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    n = len(y_true)

    values = []
    for _ in range(n_bootstrap):
        idx = rng.randint(0, n, n)
        y_true_bs = y_true[idx]
        y_score_bs = y_score[idx]
        if len(np.unique(y_true_bs)) < 2:
            continue  # bootstrap sem as duas classes -> métrica indefinida, pula
        values.append(metric_fn(y_true_bs, y_score_bs))

    values = np.array(values)
    alpha = 1 - ci
    lower = float(np.percentile(values, 100 * alpha / 2))
    upper = float(np.percentile(values, 100 * (1 - alpha / 2)))
    return lower, upper, float(np.mean(values))


def full_metrics_with_ci(y_true, y_score, threshold=0.5, n_bootstrap=1000, seed=42):
    """Métricas completas + IC 95% por bootstrap para AUROC e AUPRC."""
    base = compute_metrics(y_true, y_score, threshold=threshold)

    auroc_lo, auroc_hi, auroc_mean = bootstrap_ci(
        y_true, y_score, roc_auc_score, n_bootstrap=n_bootstrap, seed=seed
    )
    auprc_lo, auprc_hi, auprc_mean = bootstrap_ci(
        y_true, y_score, average_precision_score, n_bootstrap=n_bootstrap, seed=seed
    )

    base["auroc_ci95"] = [auroc_lo, auroc_hi]
    base["auprc_ci95"] = [auprc_lo, auprc_hi]
    base["bootstrap_n"] = n_bootstrap

    return base


### Conexão ao MLflow

In [19]:
import shutil, os

os.makedirs('/kaggle/working/mlflow-server', exist_ok=True)

origem = '/kaggle/input/datasets/joovitorbach/mmlflow1/mlflow(1) (1).db'
destino = '/kaggle/working/mlflow-server/mlflow.db'  # precisa bater com MLFLOW_TRACKING_URI

assert os.path.exists(origem), f"Arquivo de origem não encontrado: {origem}"
shutil.copy(origem, destino)

print(f"Histórico do MLflow restaurado de '{origem}' para '{destino}'.")
print(f"Tamanho copiado: {os.path.getsize(destino)} bytes.")

Histórico do MLflow restaurado de '/kaggle/input/datasets/joovitorbach/mmlflow1/mlflow(1) (1).db' para '/kaggle/working/mlflow-server/mlflow.db'.
Tamanho copiado: 1458176 bytes.


In [20]:
# ============================================================
# Conexão ao MLflow — segue o mesmo padrão do restante do grupo
# (scripts/07_train_baseline.py, scripts/08_mlflow_smoke_test.py):
# grava direto em um banco SQLite local, SEM precisar de um servidor
# remoto compartilhado rodando. Isso funciona igual dentro do Kaggle.
#
# Se, no futuro, o grupo decidir usar um servidor MLflow remoto de
# verdade (não localhost — teria que ser uma URL pública), basta
# cadastrar o Secret MLFLOW_TRACKING_URI que este notebook já usa ele
# automaticamente no lugar do SQLite local.
# ============================================================
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mlflow"], check=True)

import mlflow

MLFLOW_TRACKING_URI = None
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    MLFLOW_TRACKING_URI = secrets.get_secret("MLFLOW_TRACKING_URI")
    if secrets.get_secret("MLFLOW_TRACKING_TOKEN"):
        os.environ["MLFLOW_TRACKING_TOKEN"] = secrets.get_secret("MLFLOW_TRACKING_TOKEN")
except Exception:
    MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI")

if not MLFLOW_TRACKING_URI:
    # Sem servidor remoto configurado -> grava local, dentro do próprio
    # ambiente do Kaggle. O arquivo mlflow.db fica em /kaggle/working e
    # deve ser baixado ao final da sessão (ele NÃO é salvo automaticamente
    # depois que o notebook desliga).
    os.makedirs("/kaggle/working/mlflow-server", exist_ok=True)
    MLFLOW_TRACKING_URI = "sqlite:////kaggle/working/mlflow-server/mlflow.db"
    print("[mlflow] Nenhum servidor remoto configurado (Secret MLFLOW_TRACKING_URI ausente).")
    print(f"[mlflow] Gravando localmente em: {MLFLOW_TRACKING_URI}")
    print("[mlflow] IMPORTANTE: baixe a pasta /kaggle/working/mlflow-server ao final "
          "da sessão (Output do notebook), ou os runs somem quando a sessão encerrar.")
else:
    print(f"[mlflow] Usando servidor remoto configurado: {MLFLOW_TRACKING_URI}")

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Nome do experimento segue a convenção <grupo>-<tarefa>-<eixo-de-variacao>
EXPERIMENT_NAME_SMOKE = "g1-classificacao-smoke-test"
EXPERIMENT_NAME_FULL = f"g1-classificacao-{CONFIG['ablation_axis']}"

mlflow.set_experiment(EXPERIMENT_NAME_SMOKE)
print(f"Experimento (smoke test): {EXPERIMENT_NAME_SMOKE}")
print(f"Experimento (run completo): {EXPERIMENT_NAME_FULL}")


[mlflow] Usando servidor remoto configurado: sqlite:////kaggle/working/mlflow-server/mlflow.db
Experimento (smoke test): g1-classificacao-smoke-test
Experimento (run completo): g1-classificacao-resolution


### Modelo e lógica de execução de um run (compartilhada entre smoke test e run completo)

In [21]:
import torch
import torch.nn as nn
import subprocess
from torch.utils.data import DataLoader
from torchvision import models

# Caminho do script de avaliação compartilhado do grupo — ajuste se o seu
# dataset Kaggle tiver um nome diferente.
EVALUATE_PATH = "/kaggle/input/datasets/joovitorbach/evaluate-py-compartilhado/evaluate_shared/evaluate_shared"


def build_model(architecture, pretrained, num_classes=1):
    if architecture != "resnet50":
        raise NotImplementedError(f"Arquitetura '{architecture}' não implementada.")
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels, _pid, _path in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().flatten().tolist())
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
    metrics = compute_metrics(all_labels, all_scores)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def run_shared_evaluate(predictions_df: "pd.DataFrame", run_id: str) -> dict:
    """Chama o evaluate.py compartilhado do grupo em vez de calcular as
    métricas com lógica própria — garante que a avaliação usa exatamente
    o mesmo protocolo que os outros grupos/sprints, sem escolher limiar
    no teste (isso é responsabilidade do evaluate.py, não daqui)."""
    eval_input_df = predictions_df.rename(
        columns={"patient_id": "case_id", "true_label": "y_true", "predicted_probability": "y_score"}
    )[["case_id", "y_true", "y_score"]]

    preds_path = os.path.join(WORKDIR, f"{run_id}_preds_for_eval.csv")
    metrics_out_path = os.path.join(WORKDIR, f"{run_id}_eval_metrics.json")
    eval_input_df.to_csv(preds_path, index=False)

    cmd = ["python", f"{EVALUATE_PATH}/evaluate.py",
           "--task", "classification", "--input", preds_path, "--output", metrics_out_path]
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0 or not os.path.exists(metrics_out_path):
        print("[evaluate.py] FALHOU — stdout/stderr abaixo. Usando métricas internas como fallback.")
        print(result.stdout)
        print(result.stderr)
        os.remove(preds_path)
        return None

    with open(metrics_out_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)

    # Loga o CSV de entrada e o metrics.json de saída como artefatos, para
    # rastreabilidade completa de qual avaliação gerou qual número.
    mlflow.log_artifact(preds_path)
    mlflow.log_artifact(metrics_out_path)
    os.remove(preds_path)
    os.remove(metrics_out_path)

    return metrics


def log_metrics_dict_to_mlflow(metrics: dict, prefix: str = ""):
    """Loga qualquer dict de métricas no MLflow, achatando valores aninhados
    (ex: listas de intervalo de confiança viram _low/_high). Genérico de
    propósito, pois não sabemos de antemão todas as chaves exatas que o
    evaluate.py compartilhado retorna."""
    for key, value in metrics.items():
        full_key = f"{prefix}{key}"
        if isinstance(value, bool):
            continue
        elif isinstance(value, (int, float)):
            mlflow.log_metric(full_key, value)
        elif isinstance(value, (list, tuple)) and len(value) == 2 and all(isinstance(v, (int, float)) for v in value):
            mlflow.log_metric(f"{full_key}_low", value[0])
            mlflow.log_metric(f"{full_key}_high", value[1])
        elif isinstance(value, dict):
            log_metrics_dict_to_mlflow(value, prefix=f"{full_key}_")
        # strings, listas de tamanho != 2, matrizes de confusão completas etc.
        # ficam só no metrics.json (artefato), não como mlflow.log_metric.


def execute_run(config: dict, smoke_test: bool, mlflow_experiment: str,
                 max_smoke_samples: int = 16, extra_tags: dict = None):
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"

    mlflow.set_experiment(mlflow_experiment)

    with mlflow.start_run(run_name=f"{config['experiment_id']}{'-smoke' if smoke_test else ''}") as run:
        mlflow.set_tags({
            "owner": config["owner"],
            "role": config["role"],
            "status": "exploratory" if smoke_test else "success",  # sobrescrito no fim/erro
            "ablation_axis": "none" if smoke_test else config["ablation_axis"],
            "environment": ENVIRONMENT,
            **(extra_tags or {}),
        })

        mlflow.log_params({
            "code_version": CODE_VERSION,
            "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"],
            "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"],
            "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"],
            "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"],
            "preprocessing_version": config["preprocessing_version"],
            "environment": ENVIRONMENT,
        })

        try:
            print("Construindo mapa de tradução dicom_info.csv -> jpeg real...")
            dicom_lookup = build_dicom_info_lookup(config["dataset_root"])
            print(f"Mapa construído: {len(dicom_lookup)} imagens mapeadas.")

            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])

            if smoke_test:
                train_df = train_df.head(max_smoke_samples)
                val_df = val_df.head(max(4, max_smoke_samples // 4))

            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=not smoke_test,
                                             dicom_lookup=dicom_lookup)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup)

            # num_workers>0 paraleliza a leitura/decodificação de imagem, que é
            # o gargalo real (não a GPU) — sem isso a GPU fica ociosa esperando
            # dado. pin_memory acelera a transferência CPU->GPU quando há CUDA.
            NUM_WORKERS = 2
            train_loader = DataLoader(train_ds, batch_size=min(config["batch_size"], len(train_ds)),
                                       shuffle=True, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                       persistent_workers=NUM_WORKERS > 0)
            val_loader = DataLoader(val_ds, batch_size=min(config["batch_size"], len(val_ds)),
                                     shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                     persistent_workers=NUM_WORKERS > 0)

            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            n_epochs = 1 if smoke_test else config["epochs"]
            history = []
            t_start = time.time()
            best_val_auroc, checkpoint_path = 0.0, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")

            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:
                    torch.cuda.reset_peak_memory_stats(device)

                train_m = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                val_m = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0

                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None

                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("train_auroc", train_m["auroc"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                if peak_vram_mb is not None:
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)

                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "train_auroc": train_m["auroc"], "val_auroc": val_m["auroc"],
                                 "seconds": dt, "peak_vram_mb": peak_vram_mb})

                vram_str = f"{peak_vram_mb:.0f}MB" if peak_vram_mb is not None else "N/A (CPU)"
                print(f"Época {epoch}/{n_epochs} | val_auroc={val_m['auroc']:.4f} | {dt:.1f}s | VRAM pico: {vram_str}")

                if val_m["auroc"] > best_val_auroc:
                    best_val_auroc = val_m["auroc"]
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "image_size": config["image_size"], "seed": config["seed"],
                                "experiment_id": config["experiment_id"]}, checkpoint_path)

            duration_seconds = sum(h["seconds"] for h in history)
            avg_epoch_seconds = duration_seconds / len(history) if history else None
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]

            mlflow.log_metric("duration_seconds", duration_seconds)
            if avg_epoch_seconds is not None:
                mlflow.log_metric("avg_epoch_time_seconds", avg_epoch_seconds)
            if vram_values:
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))

            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)

            config_path = os.path.join(WORKDIR, f"{run.info.run_id}_config.json")
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(config, f, indent=2, ensure_ascii=False)
            mlflow.log_artifact(config_path)
            os.remove(config_path)

            if not smoke_test:
                test_df = build_labeled_dataframe(
                    config["dataset_root"], config["test_csvs"], config["malignant_labels"],
                    config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
                )
                test_df = filter_resolvable_rows(test_df, config["dataset_root"],
                                                  config["image_path_column"], dicom_lookup)
                test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                                config["image_size"], mean, std, augment=False,
                                                dicom_lookup=dicom_lookup)
                test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                                          num_workers=NUM_WORKERS, pin_memory=use_cuda)

                model.eval()
                records = []
                with torch.no_grad():
                    for images, labels, patient_ids, _paths in test_loader:
                        probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
                        for pid, lab, prob in zip(patient_ids, labels.numpy().flatten(), probs):
                            records.append({"patient_id": pid, "true_label": int(lab),
                                             "predicted_probability": float(prob)})

                predictions_df = pd.DataFrame(records)

                # ---- Avaliação via evaluate.py compartilhado (protocolo único do grupo) ----
                print("Rodando evaluate.py compartilhado...")
                shared_metrics = run_shared_evaluate(predictions_df, run.info.run_id)

                if shared_metrics is not None:
                    log_metrics_dict_to_mlflow(shared_metrics)
                    mlflow.set_tag("evaluation_source", "evaluate_shared")
                    print("Métricas do evaluate.py compartilhado registradas no MLflow.")
                else:
                    # Fallback: métricas internas, só se o script compartilhado falhar
                    metrics_test = full_metrics_with_ci(
                        predictions_df["true_label"].values, predictions_df["predicted_probability"].values,
                        n_bootstrap=1000, seed=config["seed"],
                    )
                    log_metrics_dict_to_mlflow(metrics_test)
                    mlflow.set_tag("evaluation_source", "internal_fallback")
                    print("[AVISO] evaluate.py compartilhado falhou — usadas métricas internas como fallback.")

                if os.path.exists(checkpoint_path):
                    mlflow.log_artifact(checkpoint_path)
                    os.remove(checkpoint_path)

            else:
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)

            mlflow.set_tag("status", "success")
            return run.info.run_id

        except Exception as e:
            # Runs falhos também ficam registrados (não são descartados nem
            # escondidos), conforme critério de aceite da ablação.
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            print(f"[ERRO] Run falhou, mas foi registrado no MLflow com status=failed: {e}")
            raise


In [22]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models


def build_model(architecture, pretrained, num_classes=1):
    if architecture != "resnet50":
        raise NotImplementedError(f"Arquitetura '{architecture}' não implementada.")
    weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
    model = models.resnet50(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def run_epoch(model, loader, criterion, optimizer, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels, _pid, _path in loader:
            images, labels = images.to(device), labels.to(device).unsqueeze(1)
            if train:
                optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * images.size(0)
            all_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().flatten().tolist())
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
    metrics = compute_metrics(all_labels, all_scores)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics


def execute_run(config: dict, smoke_test: bool, mlflow_experiment: str,
                 max_smoke_samples: int = 16, extra_tags: dict = None):
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"  # NOVO

    mlflow.set_experiment(mlflow_experiment)

    with mlflow.start_run(run_name=f"{config['experiment_id']}{'-smoke' if smoke_test else ''}") as run:
        mlflow.set_tags({
            "owner": config["owner"],
            "role": config["role"],
            "status": "exploratory" if smoke_test else "success",
            "ablation_axis": "none" if smoke_test else config["ablation_axis"],
            "environment": ENVIRONMENT,
            **(extra_tags or {}),
        })

        mlflow.log_params({
            "code_version": CODE_VERSION,
            "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"],
            "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"],
            "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"],
            "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"],
            "preprocessing_version": config["preprocessing_version"],
            "environment": ENVIRONMENT,
        })

        try:
            print("Construindo mapa de tradução dicom_info.csv -> jpeg real...")
            dicom_lookup = build_dicom_info_lookup(config["dataset_root"])
            print(f"Mapa construído: {len(dicom_lookup)} imagens mapeadas.")

            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])

            if smoke_test:
                train_df = train_df.head(max_smoke_samples)
                val_df = val_df.head(max(4, max_smoke_samples // 4))

            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=not smoke_test,
                                             dicom_lookup=dicom_lookup)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup)

            train_loader = DataLoader(train_ds, batch_size=min(config["batch_size"], len(train_ds)),
                                       shuffle=True, num_workers=2, pin_memory=use_cuda)  # NUM_WORKERS corrigido
            val_loader = DataLoader(val_ds, batch_size=min(config["batch_size"], len(val_ds)),
                                     shuffle=False, num_workers=2, pin_memory=use_cuda)

            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            n_epochs = 1 if smoke_test else config["epochs"]
            history = []
            t_start = time.time()
            best_val_auroc, checkpoint_path = 0.0, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")

            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:  # NOVO: reseta o contador de pico de VRAM a cada época
                    torch.cuda.reset_peak_memory_stats(device)

                train_m = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                val_m = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0

                # NOVO: captura o pico de VRAM da época
                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None

                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("train_auroc", train_m["auroc"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                if peak_vram_mb is not None:  # NOVO
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)

                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "train_auroc": train_m["auroc"], "val_auroc": val_m["auroc"],
                                 "seconds": dt, "peak_vram_mb": peak_vram_mb})  # NOVO: peak_vram_mb

                vram_str = f"{peak_vram_mb:.0f}MB" if peak_vram_mb is not None else "N/A (CPU)"
                print(f"Época {epoch}/{n_epochs} | val_auroc={val_m['auroc']:.4f} | {dt:.1f}s | VRAM pico: {vram_str}")

                if val_m["auroc"] > best_val_auroc:
                    best_val_auroc = val_m["auroc"]
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "image_size": config["image_size"], "seed": config["seed"],
                                "experiment_id": config["experiment_id"]}, checkpoint_path)

            duration_seconds = sum(h["seconds"] for h in history)
            avg_epoch_time_seconds = duration_seconds / len(history) if history else None  # NOVO
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]  # NOVO

            mlflow.log_metric("duration_seconds", duration_seconds)
            if avg_epoch_time_seconds is not None:  # NOVO
                mlflow.log_metric("avg_epoch_time_seconds", avg_epoch_time_seconds)
            if vram_values:  # NOVO
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:  # NOVO
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))

            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)

            config_path = os.path.join(WORKDIR, f"{run.info.run_id}_config.json")
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(config, f, indent=2, ensure_ascii=False)
            mlflow.log_artifact(config_path)
            os.remove(config_path)

            if not smoke_test:
                test_df = build_labeled_dataframe(
                    config["dataset_root"], config["test_csvs"], config["malignant_labels"],
                    config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
                )
                test_df = filter_resolvable_rows(test_df, config["dataset_root"],
                                                  config["image_path_column"], dicom_lookup)
                test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                                config["image_size"], mean, std, augment=False,
                                                dicom_lookup=dicom_lookup)
                test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                                          num_workers=2, pin_memory=use_cuda)

                model.eval()
                records = []
                with torch.no_grad():
                    for images, labels, patient_ids, _paths in test_loader:
                        probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
                        for pid, lab, prob in zip(patient_ids, labels.numpy().flatten(), probs):
                            records.append({"patient_id": pid, "true_label": int(lab),
                                             "predicted_probability": float(prob)})

                predictions_df = pd.DataFrame(records)
                metrics_test = full_metrics_with_ci(
                    predictions_df["true_label"].values, predictions_df["predicted_probability"].values,
                    n_bootstrap=1000, seed=config["seed"],
                )

                mlflow.log_metrics({
                    "auroc": metrics_test["auroc"], "auprc": metrics_test["auprc"],
                    "sensitivity_at_90_specificity": metrics_test["sensitivity_at_specificity_90"],
                    "specificity_at_90_sensitivity": metrics_test["specificity_at_sensitivity_90"],
                    "f1": metrics_test["f1_at_threshold_0.5"],
                    "bootstrap_auroc_lower": metrics_test["auroc_ci95"][0],
                    "bootstrap_auroc_upper": metrics_test["auroc_ci95"][1],
                    "bootstrap_auprc_lower": metrics_test["auprc_ci95"][0],
                    "bootstrap_auprc_upper": metrics_test["auprc_ci95"][1],
                })
                tn, fp, fn, tp = 0, 0, 0, 0
                cm = metrics_test["confusion_matrix_at_threshold_0.5"]
                if len(cm) == 2:
                    tn, fp = cm[0]
                    fn, tp = cm[1]
                mlflow.log_metrics({"tp": tp, "fp": fp, "tn": tn, "fn": fn})

                metrics_path = os.path.join(WORKDIR, f"{run.info.run_id}_metrics.json")
                with open(metrics_path, "w", encoding="utf-8") as f:
                    json.dump(metrics_test, f, indent=2, ensure_ascii=False)
                mlflow.log_artifact(metrics_path)
                os.remove(metrics_path)

                preds_path = os.path.join(WORKDIR, f"{run.info.run_id}_predictions.csv")
                predictions_df.to_csv(preds_path, index=False)
                mlflow.log_artifact(preds_path)
                os.remove(preds_path)

                if os.path.exists(checkpoint_path):
                    mlflow.log_artifact(checkpoint_path)
                    os.remove(checkpoint_path)

                print(f"AUROC teste: {metrics_test['auroc']:.4f} "
                      f"(IC95%: {metrics_test['auroc_ci95'][0]:.4f}-{metrics_test['auroc_ci95'][1]:.4f})")
            else:
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)

            mlflow.set_tag("status", "success")
            return run.info.run_id

        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            raise

## Smoke test (obrigatório antes do run completo)

Roda uma versão reduzida (poucas imagens, 1 época) só para validar que o
pipeline inteiro funciona neste ambiente — dataset carrega, modelo treina,
MLflow registra. Tag `status=exploratory`, fica isolado no experimento
`g1-classificacao-smoke-test`, nunca misturado com resultados de comparação
real.

In [ ]:
import torch
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Nome da GPU:", torch.cuda.get_device_name(0))
    print("Capacidade:", torch.cuda.get_device_capability(0))
print("=== SMOKE TEST — validando o pipeline antes do run completo ===")
smoke_run_id = execute_run(
    CONFIG, smoke_test=True, mlflow_experiment=EXPERIMENT_NAME_SMOKE,
    max_smoke_samples=16,
)
print(f"\nSmoke test concluído com sucesso. Run ID: {smoke_run_id}")
print("Pipeline validado — pode prosseguir para o run completo na próxima célula.")


## Run completo

**Só rode esta célula depois que o smoke test acima terminou sem erro.**
Treina com a configuração completa de `CONFIG`, avalia no teste oficial
congelado, e registra tudo no MLflow (experimento
`g1-classificacao-<eixo>`).

In [ ]:
# Só chegue até aqui depois que a célula do smoke test rodou sem erro.
print("=== RUN COMPLETO ===")
full_run_id = execute_run(
    CONFIG, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_FULL,
)
print(f"\nRun completo concluído. Run ID: {full_run_id}")
print(f"Confira os resultados no MLflow, experimento '{EXPERIMENT_NAME_FULL}'.")


In [ ]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("g1-classificacao-resolution")  # experimento do EXP-01
runs = client.search_runs([exp.experiment_id], order_by=["start_time ASC"])

for r in runs:
    print(r.info.run_id, "|", r.data.tags.get("mlflow.runName"), "|",
          "status:", r.data.tags.get("status"), "|",
          "seed:", r.data.params.get("seed"), "|",
          "auroc:", r.data.metrics.get("auroc"))

## Bacth size 4 configs

In [ ]:
# ============================================================
# Configurações versionadas da ablação de batch size.
# Cada uma é uma cópia EXATA do CONFIG do EXP-01 (baseline), mudando
# APENAS batch_size e experiment_id — nenhuma outra variável muda,
# conforme o critério de aceite desta sprint.
# ============================================================

CONFIG_BATCH_08 = {**CONFIG, "experiment_id": "EXP-13", "batch_size": 8, "ablation_axis": "batch_size"}
CONFIG_BATCH_16 = {**CONFIG, "experiment_id": "EXP-14", "batch_size": 16, "ablation_axis": "batch_size"}
CONFIG_BATCH_32 = {**CONFIG, "experiment_id": "EXP-15", "batch_size": 32, "ablation_axis": "batch_size"}
CONFIG_BATCH_64 = {**CONFIG, "experiment_id": "EXP-16", "batch_size": 64, "ablation_axis": "batch_size"}

BATCH_CONFIGS = [CONFIG_BATCH_08, CONFIG_BATCH_16, CONFIG_BATCH_32, CONFIG_BATCH_64]

# Conferência rápida: só batch_size deve diferir entre eles
campos_para_conferir = ["seed", "architecture", "pretrained_weights", "image_size",
                         "learning_rate", "weight_decay", "epochs", "dataset_root"]
for campo in campos_para_conferir:
    valores = {c["experiment_id"]: c[campo] for c in BATCH_CONFIGS}
    if len(set(valores.values())) > 1:
        raise RuntimeError(f"Campo '{campo}' difere entre os configs: {valores} "
                            f"— isso quebra o critério de 'só batch size muda'.")
print("Conferência OK: todos os configs são idênticos exceto batch_size.")
for c in BATCH_CONFIGS:
    print(f"  {c['experiment_id']}: batch_size={c['batch_size']}")


In [ ]:
# ============================================================
# Roda os 4 experimentos da ablação. Cada um em seu próprio try/except
# — se um falhar, os outros continuam rodando (runs falhos ficam
# registrados no MLflow com status=failed, não travam a sprint inteira).
# ============================================================

EXPERIMENT_NAME_BATCH = "g1-classificacao-batch_size"
batch_run_ids = {}

for cfg in BATCH_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Rodando {cfg['experiment_id']} (batch_size={cfg['batch_size']})")
    print(f"{'='*60}")
    try:
        run_id = execute_run(cfg, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_BATCH)
        batch_run_ids[cfg["experiment_id"]] = {"run_id": run_id, "batch_size": cfg["batch_size"], "status": "success"}
        print(f"OK — {cfg['experiment_id']} concluído. run_id: {run_id}")
    except Exception as e:
        batch_run_ids[cfg["experiment_id"]] = {"run_id": None, "batch_size": cfg["batch_size"],
                                                 "status": "failed", "error": str(e)[:200]}
        print(f"FALHOU — {cfg['experiment_id']} (batch_size={cfg['batch_size']}): {e}")
        print("Registrado no MLflow com status=failed. Seguindo para o próximo batch size.")

print("\n\n=== RESUMO DA ABLAÇÃO ===")
for exp_id, info in batch_run_ids.items():
    print(f"  {exp_id} (batch={info['batch_size']}): {info['status']} | run_id: {info['run_id']}")


In [ ]:
# ============================================================
# Manifesto consolidado da ablação — reúne tudo que a issue de
# "coleta de resultados" pede num único arquivo exportável, sem
# precisar que quem avalia entre no MLflow run por run.
# ============================================================

client = mlflow.tracking.MlflowClient()

manifest_rows = []
for exp_id, info in batch_run_ids.items():
    row = {
        "experiment_id": exp_id,
        "run_id": info["run_id"] or "N/A",
        "status": info["status"],
        "batch_size": info["batch_size"],
    }

    if info["run_id"]:
        run = client.get_run(info["run_id"])
        p = run.data.params
        m = run.data.metrics
        row.update({
            # commit/configuração
            "code_version": p.get("code_version", "NAO DEFINIDO"),
            "seed": p.get("seed"),
            "architecture": p.get("architecture"),
            "pretrained_weights": p.get("pretrained_weights"),
            "image_size": p.get("image_size"),
            "learning_rate": p.get("learning_rate"),
            "weight_decay": p.get("weight_decay"),
            "epochs": p.get("epochs"),
            # dataset/manifest e split
            "dataset_manifest": p.get("dataset_manifest"),
            "split_strategy": p.get("split_strategy"),
            # ambiente
            "environment": p.get("environment"),
            "gpu_name": p.get("gpu_name", "N/A"),
            # custo
            "avg_epoch_time_seconds": m.get("avg_epoch_time_seconds"),
            "peak_vram_mb_max": m.get("peak_vram_mb_max"),
            "duration_seconds": m.get("duration_seconds"),
            # avaliação
            "evaluation_source": run.data.tags.get("evaluation_source", "N/A"),
            "auroc": m.get("auroc"),
            # artefatos externos referenciados (não embutidos, só a referência)
            "dataset_source_kaggle": CONFIG["dataset_root"],
            "evaluate_script_path": EVALUATE_PATH,
        })
    else:
        row["error_message"] = info.get("error", "")

    manifest_rows.append(row)

manifest_df = pd.DataFrame(manifest_rows).sort_values("batch_size")

# Aviso explícito se algum code_version não estiver definido — é critério
# de aceite desta sprint, não pode passar batido.
sem_code_version = manifest_df[manifest_df.get("code_version", "") == "NAO DEFINIDO"]
if len(sem_code_version) > 0:
    print("[ATENÇÃO] Os seguintes runs NÃO têm code_version definido — "
          "corrija antes de considerar a sprint concluída:")
    print(sem_code_version["experiment_id"].tolist())

manifest_path = os.path.join(WORKDIR, "manifesto_resultados_batch_ablation.csv")
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifesto salvo em: {manifest_path}")
print(manifest_df.to_string(index=False))

# Registra o manifesto como artefato também, num run "documentação" próprio
mlflow.set_experiment(EXPERIMENT_NAME_BATCH)
with mlflow.start_run(run_name="EXP-batch-ablation-manifesto"):
    mlflow.set_tags({"owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
                      "run_type": "results_manifest", "environment": ENVIRONMENT})
    mlflow.log_artifact(manifest_path)

print("\nManifesto também registrado no MLflow (run 'EXP-batch-ablation-manifesto').")


In [ ]:
import os

print("=== Tudo que está em /kaggle/input/datasets/ ===")
base = "/kaggle/input/datasets"
if os.path.isdir(base):
    for root, dirs, files in os.walk(base):
        depth = root.replace(base, '').count(os.sep)
        if depth > 3:
            dirs[:] = []
            continue
        print(root)
        for f in files:
            print("   ", f)
else:
    print("Pasta /kaggle/input/datasets/ nem existe.")

In [ ]:
# ============================================================
# Comparação com o EXP-01 original usando valores copiados do LOG da
# versão commitada antiga (o MLflow desta sessão nova está vazio,
# então não dá pra buscar por run_id — usamos os números direto).
# ============================================================
from pathlib import Path
import json

THRESHOLDS = {
    "auroc": 0.015,
    "auprc": 0.015,
    "sensitivity_at_90_specificity": 0.03,
}


def compare_metric(label, val_a, val_b, threshold, relative=False):
    if val_a is None or val_b is None:
        return None, f"[AVISO] '{label}' ausente em uma das execuções (A={val_a}, B={val_b})."
    diff = val_b - val_a
    if relative:
        denom = val_a if val_a != 0 else 1e-9
        rel_diff = abs(diff) / abs(denom)
        flagged = rel_diff > threshold
        msg = (f"{label}: A={val_a:.4f} | B={val_b:.4f} | diferença relativa={rel_diff*100:.1f}% "
               f"{'>>> DIVERGÊNCIA' if flagged else '(dentro do esperado)'}")
    else:
        flagged = abs(diff) > threshold
        msg = (f"{label}: A={val_a:.4f} | B={val_b:.4f} | diferença={diff:+.4f} "
               f"{'>>> DIVERGÊNCIA' if flagged else '(dentro do esperado)'}")
    return flagged, msg

# EDITE AQUI: copie os valores impressos no log da versão antiga do EXP-01.
# Números SEM aspas e SEM unidade (ex: 3064, não "3064MB") — senão a
# comparação matemática quebra. Deixe None só quando não tiver o dado.
ORIGINAL_EXP01_VALUES = {
    "seed": "42",  # como string, igual o MLflow guarda os params
    "architecture": "resnet50",
    "batch_size": "32",
    "learning_rate": "0.0001",
    "image_size": "224",
    "auroc": 0.690,          # <-- único valor obrigatório (é o único impresso no log)
    "auprc": None,           # geralmente NÃO aparece no log (só vai pro MLflow) — ok deixar None
    "sensitivity_at_90_specificity": None,  # idem — ok deixar None
    "avg_epoch_time_seconds": None,
    "peak_vram_mb_max": 3064.0,  # sem "MB" — só o número
    "gpu_name": "Tesla T4",
}

for campo in ("auroc",):  # só o AUROC é realmente obrigatório (é o único impresso no log)
    if ORIGINAL_EXP01_VALUES.get(campo) is None:
        raise ValueError(f"Preencha '{campo}' em ORIGINAL_EXP01_VALUES — é o único campo obrigatório.")



def compare_with_manual_baseline(baseline_values: dict, rerun_run_id: str):
    """Mesma lógica de compare_runs_mlflow, mas o lado A vem de valores
    digitados manualmente (recuperados do log), não de uma consulta ao MLflow."""
    client = mlflow.tracking.MlflowClient()
    run_b = client.get_run(rerun_run_id)

    report_lines = ["=" * 70, "COMPARAÇÃO — REEXECUÇÃO vs. EXP-01 ORIGINAL (valores manuais do log)",
                     "=" * 70, f"Run B (reexecução): {rerun_run_id}", ""]

    report_lines.append("--- Configuração ---")
    config_issues = []
    for campo in ["seed", "architecture", "batch_size", "learning_rate", "image_size"]:
        val_a, val_b = baseline_values.get(campo), run_b.data.params.get(campo)
        if str(val_a) != str(val_b):
            config_issues.append(f"[CRÍTICO] '{campo}' difere: A='{val_a}' vs B='{val_b}'.")
    report_lines.extend(config_issues or ["[OK] Configuração bate com o log original."])
    report_lines.append("")

    report_lines.append("--- Métricas no teste congelado ---")
    divergences = []
    for metric_key, label in [("auroc", "AUROC"), ("auprc", "AUPRC"),
                                ("sensitivity_at_90_specificity", "Sensibilidade @ especificidade 90%")]:
        val_a = baseline_values.get(metric_key)
        val_b = run_b.data.metrics.get(metric_key)
        flagged, msg = compare_metric(label, val_a, val_b, THRESHOLDS.get(metric_key, 0.02))
        report_lines.append(msg)
        if flagged:
            divergences.append(msg)
    report_lines.append("")

    report_lines.append("--- Resumo ---")
    if divergences:
        report_lines.append(f"{len(divergences)} divergência(s) acima do limiar:")
        report_lines.extend(f"  - {d}" for d in divergences)
    else:
        report_lines.append("Nenhuma divergência acima do limiar — consistente com o EXP-01 original.")

    report_text = "\n".join(report_lines)
    print(report_text)
    return {"baseline_values": baseline_values, "rerun_run_id": rerun_run_id,
            "config_issues": config_issues, "divergences": divergences, "report_text": report_text}


# Reexecuta com a MESMA configuração/seed do original
CONFIG_REEXECUCAO = {**CONFIG, "experiment_id": "EXP-01-rerun", "ablation_axis": "reproducibility"}
rerun_run_id = execute_run(CONFIG_REEXECUCAO, smoke_test=False, mlflow_experiment="g1-classificacao-resolution")

comparison_result = compare_with_manual_baseline(ORIGINAL_EXP01_VALUES, rerun_run_id)

comparison_path = Path(WORKDIR) / "comparacao_reexecucao.json"
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(comparison_result, f, indent=2, ensure_ascii=False)
print(f"\nComparação registrada em: {comparison_path}")

## **Sprint 6 cria as 3 seeds e faz a comparação com cada**

In [ ]:
import matplotlib.cm as cm


class GradCAM:
    """Grad-CAM padrão para ResNet50: usa o último bloco convolucional
    (layer4) como camada alvo — é a escolha usual para essa arquitetura.
    """

    def __init__(self, model, target_layer=None):
        self.model = model
        self.target_layer = target_layer or model.layer4[-1]
        self.gradients = None
        self.activations = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0].detach()

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate(self, input_tensor):
        """input_tensor: (1, 3, H, W). Retorna um mapa de calor (H, W) em [0,1]."""
        self.model.zero_grad()
        logit = self.model(input_tensor)  # saída única (logit de malignidade)
        logit.backward()

        # Pesos = média global dos gradientes por canal (GAP), como no Grad-CAM original
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, h, w)
        cam = torch.relu(cam)

        cam = cam.squeeze().cpu().numpy()
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        else:
            cam = np.zeros_like(cam)
        return cam


def save_sanitized_gradcam(model, image_tensor, image_size, sample_index, output_dir, device, alpha=0.45):
    """Gera e salva o overlay Grad-CAM de UMA amostra, com nome de arquivo
    SANITIZADO (só um índice sequencial, nunca patient_id/nome de arquivo
    original — evita vazar identificador de paciente em artefato salvo).
    """
    model.eval()
    gradcam = GradCAM(model)

    # Precisa de gradiente habilitado só pro forward/backward do Grad-CAM,
    # mesmo com o modelo em eval() (BatchNorm/Dropout continuam em modo eval).
    with torch.enable_grad():
        input_for_cam = image_tensor.unsqueeze(0).to(device).clone().requires_grad_(True)
        cam = gradcam.generate(input_for_cam)


    cam_resized = np.array(
        Image.fromarray((cam * 255).astype(np.uint8)).resize((image_size, image_size), Image.BILINEAR)
    ) / 255.0

    # Desfaz a normalização ImageNet só para exibição (não altera o tensor usado no modelo)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_np = image_tensor.cpu().numpy().transpose(1, 2, 0)
    img_np = np.clip(img_np * std + mean, 0, 1)
    img_uint8 = (img_np * 255).astype(np.uint8)

    colored = (cm.jet(cam_resized)[:, :, :3] * 255).astype(np.uint8)
    overlay = (img_uint8.astype(np.float32) * (1 - alpha) + colored.astype(np.float32) * alpha).astype(np.uint8)

    fname = f"gradcam_sample_{sample_index:02d}.png"  # nome sanitizado, sem patient_id
    out_path = Path(output_dir) / fname
    Image.fromarray(overlay).save(out_path)
    return out_path

In [ ]:
# ============================================================
# Configurações versionadas para o estudo de estabilidade entre seeds.
# Arquitetura, manifesto, split, pré-processamento e hiperparâmetros
# CONGELADOS (idênticos ao EXP-01) — só a seed muda entre as 3 execuções.
# ============================================================

SEED_STABILITY_SEEDS = [42, 7, 123]

CONFIG_SEED_42 = {**CONFIG, "experiment_id": "EXP-05-seed42", "seed": 42, "ablation_axis": "seed"}
CONFIG_SEED_07 = {**CONFIG, "experiment_id": "EXP-05-seed07", "seed": 7, "ablation_axis": "seed"}
CONFIG_SEED_123 = {**CONFIG, "experiment_id": "EXP-05-seed123", "seed": 123, "ablation_axis": "seed"}

SEED_STABILITY_CONFIGS = [CONFIG_SEED_42, CONFIG_SEED_07, CONFIG_SEED_123]

# Conferência automática: tudo igual entre as 3, exceto seed/experiment_id
campos_para_conferir = ["architecture", "pretrained_weights", "image_size", "batch_size",
                         "learning_rate", "weight_decay", "epochs", "dataset_root",
                         "train_csvs", "test_csvs"]
for campo in campos_para_conferir:
    valores = [c[campo] for c in SEED_STABILITY_CONFIGS]
    if any(v != valores[0] for v in valores):
        raise RuntimeError(
            f"Campo '{campo}' difere entre as configs de seed: {valores} — isso quebra "
            f"o critério de 'a seed é a única variável deliberadamente alterada'."
        )
print("Conferência OK: configuração congelada (arquitetura/manifesto/split/hiperparâmetros) "
      "idêntica nas 3 seeds.")
for c in SEED_STABILITY_CONFIGS:
    print(f"  {c['experiment_id']}: seed={c['seed']}")

In [ ]:
EXPERIMENT_NAME_SEED_STABILITY = "g1-classificacao-seed_stability"
N_GRADCAM_SAMPLES = 6  # quantas amostras de validação recebem Grad-CAM salvo


def execute_seed_stability_run(config: dict, mlflow_experiment: str):
    """Treina e avalia SOMENTE em treino/validação — o conjunto de teste
    NUNCA é tocado aqui, conforme exigido pela sprint (o teste só é
    liberado depois que a configuração for congelada, numa etapa futura
    separada). Gera Grad-CAM sanitizado de algumas amostras de validação.
    """
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"

    mlflow.set_experiment(mlflow_experiment)

    with mlflow.start_run(run_name=config["experiment_id"]) as run:
        mlflow.set_tags({
            "owner": config["owner"], "role": config["role"], "status": "success",
            "ablation_axis": "seed", "environment": ENVIRONMENT,
            "test_set_touched": "false",  # crítico: registra explicitamente que o teste não foi usado
        })
        mlflow.log_params({
            "code_version": CODE_VERSION, "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"], "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"], "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"], "weight_decay": config["weight_decay"],
            "epochs": config["epochs"], "preprocessing_version": config["preprocessing_version"],
            "environment": ENVIRONMENT,
        })

        try:
            dicom_lookup = build_dicom_info_lookup(config["dataset_root"])

            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])
            # NOTA: test_csvs nunca são lidos nesta função — o teste fica intocado por construção,
            # não só por convenção.

            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=True,
                                             dicom_lookup=dicom_lookup)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup)

            train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True,
                                       num_workers=2, pin_memory=use_cuda, persistent_workers=True)
            val_loader = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False,
                                     num_workers=2, pin_memory=use_cuda)

            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

            history = []
            t_start = time.time()
            best_val_metrics, checkpoint_path = None, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")

            for epoch in range(1, config["epochs"] + 1):
                t0 = time.time()
                if use_cuda:
                    torch.cuda.reset_peak_memory_stats(device)

                train_m = run_epoch(model, train_loader, criterion, optimizer, device, train=True)
                val_m = run_epoch(model, val_loader, criterion, optimizer, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0
                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None

                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                mlflow.log_metric("val_sensitivity_at_spec90", val_m["sensitivity_at_specificity_90"], step=epoch)
                if peak_vram_mb is not None:
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)

                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "val_auroc": val_m["auroc"], "seconds": dt, "peak_vram_mb": peak_vram_mb})
                print(f"[{config['experiment_id']}] Época {epoch}/{config['epochs']} | "
                      f"val_auroc={val_m['auroc']:.4f} | {dt:.1f}s")

                if best_val_metrics is None or val_m["auroc"] > best_val_metrics["auroc"]:
                    best_val_metrics = val_m
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "seed": config["seed"]}, checkpoint_path)

            duration_seconds = sum(h["seconds"] for h in history)
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]
            mlflow.log_metric("duration_seconds", duration_seconds)
            mlflow.log_metric("avg_epoch_time_seconds", duration_seconds / len(history))
            if vram_values:
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))

            # ---- Métricas FINAIS de validação (não teste) com IC 95% por bootstrap ----
            model.eval()
            val_records = []
            with torch.no_grad():
                for images, labels, _pid, _path in val_loader:
                    probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
                    val_records.extend(zip(labels.numpy().flatten(), probs))
            val_labels_arr = np.array([r[0] for r in val_records])
            val_probs_arr = np.array([r[1] for r in val_records])

            val_metrics_final = full_metrics_with_ci(val_labels_arr, val_probs_arr,
                                                       n_bootstrap=1000, seed=config["seed"])
            for key in ["auroc", "auprc", "sensitivity_at_specificity_90"]:
                mlflow.log_metric(f"final_val_{key}", val_metrics_final[key])
            mlflow.log_metric("final_val_auroc_ci_low", val_metrics_final["auroc_ci95"][0])
            mlflow.log_metric("final_val_auroc_ci_high", val_metrics_final["auroc_ci95"][1])

            # ---- Grad-CAM sanitizado em algumas amostras de validação ----
            gradcam_dir = Path(WORKDIR) / f"{run.info.run_id}_gradcam"
            gradcam_dir.mkdir(parents=True, exist_ok=True)
            n_samples = min(N_GRADCAM_SAMPLES, len(val_ds))
            sample_indices = np.random.RandomState(config["seed"]).choice(len(val_ds), n_samples, replace=False)
            for i, idx in enumerate(sample_indices, 1):
                img_tensor, _label, _pid, _path = val_ds[idx]
                save_sanitized_gradcam(model, img_tensor, config["image_size"], i, gradcam_dir, device)
            mlflow.log_artifacts(str(gradcam_dir), artifact_path="gradcam")
            print(f"  {n_samples} Grad-CAMs sanitizados salvos e registrados no MLflow.")

            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)
            if os.path.exists(checkpoint_path):
                mlflow.log_artifact(checkpoint_path)
                os.remove(checkpoint_path)

            mlflow.set_tag("status", "success")
            print(f"[{config['experiment_id']}] Concluído. val_auroc final: {val_metrics_final['auroc']:.4f}")
            return run.info.run_id

        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.set_tag("test_set_touched", "false")  # falhou antes de qualquer chance de tocar o teste
            mlflow.log_param("error_message", str(e)[:250])
            print(f"[{config['experiment_id']}] FALHOU (registrado no MLflow): {e}")
            raise

In [ ]:
seed_run_ids = {}

for cfg in SEED_STABILITY_CONFIGS:
    print(f"\n{'='*60}\nRodando {cfg['experiment_id']} (seed={cfg['seed']})\n{'='*60}")
    try:
        run_id = execute_seed_stability_run(cfg, EXPERIMENT_NAME_SEED_STABILITY)
        seed_run_ids[cfg["experiment_id"]] = {"run_id": run_id, "seed": cfg["seed"], "status": "success"}
    except Exception as e:
        seed_run_ids[cfg["experiment_id"]] = {"run_id": None, "seed": cfg["seed"],
                                                "status": "failed", "error": str(e)[:200]}
        print(f"Registrado como falho, seguindo para a próxima seed.")

print("\n\n=== RESUMO ===")
for exp_id, info in seed_run_ids.items():
    print(f"  {exp_id} (seed={info['seed']}): {info['status']} | run_id: {info['run_id']}")

In [ ]:
# ============================================================
# Análise de estabilidade entre seeds e recomendação de congelamento.
# Usa SOMENTE métricas de validação (final_val_*) — nunca teste.
# ============================================================
from scipy import stats

client = mlflow.tracking.MlflowClient()

METRICAS_PRIORITARIAS = ["final_val_auroc", "final_val_auprc", "final_val_sensitivity_at_specificity_90"]

seed_rows = []
for exp_id, info in seed_run_ids.items():
    row = {"experiment_id": exp_id, "seed": info["seed"], "status": info["status"],
           "run_id": info["run_id"] or "N/A"}
    if info["run_id"]:
        run = client.get_run(info["run_id"])
        m = run.data.metrics
        for metric in METRICAS_PRIORITARIAS:
            row[metric] = m.get(metric)
        row["avg_epoch_time_seconds"] = m.get("avg_epoch_time_seconds")
        row["peak_vram_mb_max"] = m.get("peak_vram_mb_max")
        row["duration_seconds"] = m.get("duration_seconds")
    else:
        row["error"] = info.get("error", "")
    seed_rows.append(row)

seed_df = pd.DataFrame(seed_rows)
print("=== Resultado por seed ===")
print(seed_df.to_string(index=False))

# ---- Tabela de estabilidade: média, desvio, amplitude, IC 95% ----
runs_ok = seed_df[seed_df["status"] == "success"]
stability_rows = []

if len(runs_ok) < 2:
    print("\n[AVISO] Menos de 2 runs bem-sucedidos — não é possível calcular estabilidade "
          "de forma significativa. Corrija as falhas antes de tirar qualquer conclusão.")
else:
    for metric in METRICAS_PRIORITARIAS:
        valores = runs_ok[metric].dropna().values
        if len(valores) < 2:
            continue
        media = float(np.mean(valores))
        desvio = float(np.std(valores, ddof=1))  # desvio amostral (n-1)
        amplitude = float(np.max(valores) - np.min(valores))

        # IC 95% via distribuição t (apropriado para n pequeno, como n=3)
        if len(valores) >= 2:
            sem = desvio / np.sqrt(len(valores))
            t_crit = stats.t.ppf(0.975, df=len(valores) - 1)
            ic_low, ic_high = media - t_crit * sem, media + t_crit * sem
        else:
            ic_low, ic_high = None, None

        stability_rows.append({
            "metrica": metric, "n_runs": len(valores), "media": media, "desvio_padrao": desvio,
            "amplitude": amplitude, "ic95_low": ic_low, "ic95_high": ic_high,
            "coef_variacao_pct": (desvio / media * 100) if media != 0 else None,
        })

    stability_df = pd.DataFrame(stability_rows)
    print("\n=== Tabela de estabilidade (média, desvio, amplitude, IC 95%) ===")
    print(stability_df.to_string(index=False))

    # ---- Recomendação fundamentada (heurística explícita, ajustável) ----
    # Limiar sugerido: desvio padrão do AUROC entre seeds abaixo de 0.015
    # (mesma ordem de grandeza da hipótese original do EXP-05) sugere
    # estabilidade suficiente para congelar a configuração.
    LIMIAR_DESVIO_AUROC = 0.015
    auroc_stats = stability_df[stability_df["metrica"] == "final_val_auroc"]

    if len(auroc_stats) > 0:
        desvio_auroc = auroc_stats.iloc[0]["desvio_padrao"]
        amplitude_auroc = auroc_stats.iloc[0]["amplitude"]

        if desvio_auroc <= LIMIAR_DESVIO_AUROC:
            recomendacao = "CONGELAR"
            justificativa = (
                f"Desvio padrão do AUROC entre seeds ({desvio_auroc:.4f}) está dentro do limiar "
                f"de estabilidade ({LIMIAR_DESVIO_AUROC}). Amplitude: {amplitude_auroc:.4f}. "
                f"A configuração é reproduzível o suficiente para ser congelada e liberar o "
                f"conjunto de teste final."
            )
        else:
            recomendacao = "NÃO CONGELAR — investigar instabilidade"
            justificativa = (
                f"Desvio padrão do AUROC entre seeds ({desvio_auroc:.4f}) EXCEDE o limiar de "
                f"estabilidade ({LIMIAR_DESVIO_AUROC}). Amplitude: {amplitude_auroc:.4f}. "
                f"A variação entre seeds é grande o suficiente para tornar comparações entre "
                f"configurações não confiáveis — qualquer diferença menor que essa variação "
                f"pode ser só ruído de inicialização, não um efeito real. Recomenda-se investigar "
                f"causas prováveis (poucos dados, batch pequeno, learning rate alto, arquitetura "
                f"sensível à inicialização) antes de congelar ou liberar o teste."
            )

        print(f"\n{'='*60}\nRECOMENDAÇÃO: {recomendacao}\n{justificativa}\n{'='*60}")

        decision_result = {
            "recomendacao": recomendacao, "justificativa": justificativa,
            "limiar_desvio_auroc": LIMIAR_DESVIO_AUROC, "desvio_auroc_observado": float(desvio_auroc),
            "amplitude_auroc_observada": float(amplitude_auroc),
            "seeds_usadas": SEED_STABILITY_SEEDS, "config_congelada": {
                k: v for k, v in CONFIG_SEED_42.items() if k not in ("seed", "experiment_id")
            },
            "teste_final_liberado": False,  # sempre False nesta etapa — decisão de congelar
                                              # não libera o teste automaticamente, é um passo separado
            "n_runs_bem_sucedidos": len(runs_ok), "n_runs_totais": len(seed_df),
        }

        decision_path = Path(WORKDIR) / "seed_stability_decisao.json"
        with open(decision_path, "w", encoding="utf-8") as f:
            json.dump(decision_result, f, indent=2, ensure_ascii=False)

        stability_csv_path = Path(WORKDIR) / "tabela_estabilidade.csv"
        stability_df.to_csv(stability_csv_path, index=False)
        seed_df.to_csv(Path(WORKDIR) / "resultados_por_seed.csv", index=False)

        mlflow.set_experiment(EXPERIMENT_NAME_SEED_STABILITY)
        with mlflow.start_run(run_name="EXP-05-stability-summary"):
            mlflow.set_tags({"owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
                              "run_type": "stability_summary", "environment": ENVIRONMENT,
                              "test_set_touched": "false"})
            mlflow.log_artifact(str(decision_path))
            mlflow.log_artifact(str(stability_csv_path))
            mlflow.log_artifact(str(Path(WORKDIR) / "resultados_por_seed.csv"))
            mlflow.log_metric("desvio_auroc_entre_seeds", desvio_auroc)
            mlflow.log_metric("amplitude_auroc_entre_seeds", amplitude_auroc)

        print(f"\nDecisão registrada em: {decision_path}")
        print("IMPORTANTE: o conjunto de teste continua INTOCADO. Ele só deve ser avaliado "
              "em uma etapa separada e deliberada, depois que o time confirmar a decisão acima.")

## **Matriz consolidada de experimentos sprint 6**

In [ ]:
# ============================================================
# Matriz consolidada de experimentos — Sprint 06.
# Reúne TODOS os runs de TODOS os experimentos relevantes do MLflow
# (aprovados e falhos), normaliza unidades, e marca claramente runs
# não comparáveis ou inválidos — sem apagar nada.
# ============================================================
from pathlib import Path
import pandas as pd

# Experimentos a consolidar (ajuste se tiver outro nome/eixo usado)
EXPERIMENTOS_A_CONSOLIDAR = [
    "g1-classificacao-resolution",     # EXP-01 baseline + reexecução
    "g1-classificacao-batch_size",     # EXP-13 a EXP-16
    "g1-classificacao-seed_stability", # EXP-05-seed42/07/123
    "g1-classificacao-smoke-test",     # smoke tests (exploratórios, não comparáveis)
]

client = mlflow.tracking.MlflowClient()


def normalizar_run(run, experiment_name):
    """Extrai e normaliza os campos de um run do MLflow numa linha da matriz."""
    p, m, t = run.data.params, run.data.metrics, run.data.tags

    status = t.get("status", "desconhecido")
    ablation_axis = t.get("ablation_axis", "N/A")
    test_set_touched = t.get("test_set_touched", "true")  # default: assume que tocou o teste
    run_type = t.get("run_type", "experiment")  # distingue summary/manifest runs dos experimentos reais

    # --- Normalização de tempo: sempre em segundos ---
    duration_seconds = m.get("duration_seconds")
    avg_epoch_time_seconds = m.get("avg_epoch_time_seconds")

    # --- Normalização de memória: sempre em MB ---
    peak_vram_mb_max = m.get("peak_vram_mb_max")

    # --- Normalização de custo: GPU-horas como proxy, já que não há
    # dado de custo em $ disponível (Kaggle free tier não fatura por
    # execução) — documentado explicitamente, não é uma estimativa
    # inventada, é uma conversão direta e rastreável do tempo medido. ---
    gpu_hours = (duration_seconds / 3600) if duration_seconds is not None else None

    # --- Métricas de TESTE (só existem em runs que avaliaram teste) ---
    metric_auroc_test = m.get("auroc")
    metric_auprc_test = m.get("auprc")
    metric_sensitivity_test = m.get("sensitivity_at_90_specificity")

    # --- Métricas de VALIDAÇÃO (estudo de estabilidade, que preserva o teste) ---
    metric_auroc_val = m.get("final_val_auroc")
    metric_auprc_val = m.get("final_val_auprc")
    metric_sensitivity_val = m.get("final_val_sensitivity_at_specificity_90")

    # --- Comparabilidade ---
    is_summary_run = run_type in ("comparison_summary", "results_manifest", "reproducibility_check")
    comparable_test = (status == "success" and test_set_touched == "true"
                        and metric_auroc_test is not None and not is_summary_run)
    comparable_val = (status == "success" and metric_auroc_val is not None and not is_summary_run)

    # --- Vazamento: não inventamos nenhuma marcação sem evidência (fora do
    # escopo estimar sem evidência) — campo existe pra ser preenchido
    # manualmente pelo time SE alguém identificar um caso real depois. ---
    invalid_leakage = False
    invalid_reason = ""
    if status == "failed":
        invalid_reason = f"Run falhou: {p.get('error_message', t.get('error_message', 'motivo não registrado'))}"
    elif status == "exploratory":
        invalid_reason = "Smoke test — exploratório por definição, não entra em comparação de desempenho."
    elif is_summary_run:
        invalid_reason = f"Run de resumo/agregação ({run_type}), não é um experimento individual."

    return {
        "run_id": run.info.run_id,
        "experiment_name": experiment_name,
        "run_name": t.get("mlflow.runName", ""),
        "status": status,
        "run_type": run_type,
        "code_version": p.get("code_version", "NAO_DEFINIDO"),
        "owner": t.get("owner", p.get("owner", "")),
        "role": t.get("role", p.get("role", "")),
        "seed": p.get("seed"),
        "architecture": p.get("architecture"),
        "pretrained_weights": p.get("pretrained_weights"),
        "image_size": p.get("image_size"),
        "batch_size": p.get("batch_size"),
        "learning_rate": p.get("learning_rate"),
        "weight_decay": p.get("weight_decay"),
        "epochs": p.get("epochs"),
        "dataset_manifest": p.get("dataset_manifest"),
        "split_strategy": p.get("split_strategy"),
        "environment": t.get("environment", p.get("environment")),
        "gpu_name": p.get("gpu_name", "N/A"),
        "ablation_axis": ablation_axis,
        "test_set_touched": test_set_touched,
        "evaluation_source": t.get("evaluation_source", "N/A"),
        "metric_auroc_test": metric_auroc_test,
        "metric_auprc_test": metric_auprc_test,
        "metric_sensitivity_test": metric_sensitivity_test,
        "metric_auroc_val": metric_auroc_val,
        "metric_auprc_val": metric_auprc_val,
        "metric_sensitivity_val": metric_sensitivity_val,
        "avg_epoch_time_seconds": avg_epoch_time_seconds,
        "peak_vram_mb_max": peak_vram_mb_max,
        "duration_seconds": duration_seconds,
        "gpu_hours": gpu_hours,
        "comparable_test": comparable_test,
        "comparable_val": comparable_val,
        "invalid_leakage": invalid_leakage,
        "invalid_reason": invalid_reason,
        "mlflow_reference": f"experimento={experiment_name} | run_id={run.info.run_id}",
    }


linhas_matriz = []
experimentos_nao_encontrados = []

for exp_name in EXPERIMENTOS_A_CONSOLIDAR:
    exp = client.get_experiment_by_name(exp_name)
    if exp is None:
        experimentos_nao_encontrados.append(exp_name)
        continue
    runs = client.search_runs([exp.experiment_id], order_by=["start_time ASC"])
    for run in runs:
        linhas_matriz.append(normalizar_run(run, exp_name))

if experimentos_nao_encontrados:
    print(f"[AVISO] Experimentos não encontrados neste MLflow (histórico pode estar em outra sessão, "
          f"restaure o mlflow.db se necessário): {experimentos_nao_encontrados}")

matriz_df = pd.DataFrame(linhas_matriz)
print(f"Total de runs consolidados: {len(matriz_df)}")
if len(matriz_df) > 0:
    print(matriz_df["status"].value_counts())

# ---- Conferência: nenhuma linha sem run_id (rastreabilidade obrigatória) ----
assert matriz_df["run_id"].notna().all(), "Existe linha sem run_id — quebra o critério de rastreabilidade."
assert (matriz_df["run_id"].str.len() > 0).all(), "Existe run_id vazio."
print("OK: todas as linhas são rastreáveis a um run_id real do MLflow.")

# ---- Salva a matriz em formato leve (CSV — versionável, git-diffable) ----
matriz_path = Path(WORKDIR) / "matriz_experimentos_sprint06.csv"
matriz_df.to_csv(matriz_path, index=False)
print(f"\nMatriz salva em: {matriz_path}")

In [ ]:
# ============================================================
# Resumo comparativo, lista de runs ausentes/falhos/inválidos, e
# gráfico de custo (GPU-horas) vs. desempenho (AUROC).
# ============================================================
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# ---- 1. Resumo comparativo — SÓ runs válidos e comparáveis entram na média ----
runs_validos_teste = matriz_df[matriz_df["comparable_test"] == True]
runs_validos_val = matriz_df[matriz_df["comparable_val"] == True]

print("=== Resumo comparativo — métricas de TESTE (runs válidos e comparáveis) ===")
if len(runs_validos_teste) > 0:
    resumo_teste = runs_validos_teste.groupby("ablation_axis").agg(
        n_runs=("run_id", "count"),
        auroc_medio=("metric_auroc_test", "mean"),
        auroc_desvio=("metric_auroc_test", "std"),
        gpu_horas_media=("gpu_hours", "mean"),
    ).reset_index()
    print(resumo_teste.to_string(index=False))
else:
    print("Nenhum run com métricas de teste comparáveis ainda.")

print("\n=== Resumo comparativo — métricas de VALIDAÇÃO (ex: estabilidade de seeds) ===")
if len(runs_validos_val) > 0:
    resumo_val = runs_validos_val.groupby("ablation_axis").agg(
        n_runs=("run_id", "count"),
        auroc_medio=("metric_auroc_val", "mean"),
        auroc_desvio=("metric_auroc_val", "std"),
        gpu_horas_media=("gpu_hours", "mean"),
    ).reset_index()
    print(resumo_val.to_string(index=False))
else:
    print("Nenhum run com métricas de validação comparáveis ainda.")

# ---- 2. Lista de runs ausentes, falhos ou inválidos ----
runs_problematicos = matriz_df[
    (matriz_df["status"] != "success") | (matriz_df["invalid_reason"] != "")
][["run_id", "experiment_name", "ablation_axis", "status", "invalid_reason", "mlflow_reference"]]

print(f"\n=== Runs ausentes/falhos/inválidos: {len(runs_problematicos)} de {len(matriz_df)} total ===")
if len(runs_problematicos) > 0:
    print(runs_problematicos.to_string(index=False))

problematicos_path = Path(WORKDIR) / "runs_ausentes_falhos_invalidos.csv"
runs_problematicos.to_csv(problematicos_path, index=False)

# ---- 3. Gráfico custo (GPU-horas) vs. desempenho (AUROC) ----
fig, ax = plt.subplots(figsize=(9, 6))

plot_df = matriz_df[matriz_df["gpu_hours"].notna()].copy()
plot_df["auroc_plot"] = plot_df["metric_auroc_test"].fillna(plot_df["metric_auroc_val"])

cores_por_status = {"success": "#2E7D32", "failed": "#C62828", "exploratory": "#9E9E9E"}
for status, cor in cores_por_status.items():
    subset = plot_df[plot_df["status"] == status]
    if len(subset) == 0:
        continue
    ax.scatter(subset["gpu_hours"], subset["auroc_plot"], label=status, color=cor,
               s=80, alpha=0.8, edgecolors="black")

for _, row in plot_df.iterrows():
    if pd.notna(row["auroc_plot"]):
        ax.annotate(row["ablation_axis"], (row["gpu_hours"], row["auroc_plot"]),
                    fontsize=7, alpha=0.7, xytext=(3, 3), textcoords="offset points")

ax.set(title="Custo computacional vs. Desempenho — Sprint 06",
       xlabel="Custo (GPU-horas)", ylabel="AUROC (teste ou validação)")
ax.legend(title="Status do run")
ax.grid(alpha=0.3)
chart_path = Path(WORKDIR) / "custo_vs_desempenho.png"
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()

# ---- 4. Resumo de cobertura (o que existe vs. o que deveria existir) ----
cobertura = {
    "total_runs_consolidados": int(len(matriz_df)),
    "runs_success": int((matriz_df["status"] == "success").sum()),
    "runs_failed": int((matriz_df["status"] == "failed").sum()),
    "runs_exploratory": int((matriz_df["status"] == "exploratory").sum()),
    "runs_comparaveis_teste": int(matriz_df["comparable_test"].sum()),
    "runs_comparaveis_validacao": int(matriz_df["comparable_val"].sum()),
    "experimentos_consolidados": EXPERIMENTOS_A_CONSOLIDAR,
    "experimentos_nao_encontrados": experimentos_nao_encontrados,
    "gpu_horas_totais": float(matriz_df["gpu_hours"].sum(skipna=True)),
}

cobertura_path = Path(WORKDIR) / "resumo_cobertura.json"
with open(cobertura_path, "w", encoding="utf-8") as f:
    json.dump(cobertura, f, indent=2, ensure_ascii=False)

print(f"\n=== Resumo de cobertura ===")
for k, v in cobertura.items():
    print(f"  {k}: {v}")

# ---- 5. Registra tudo no MLflow como evidência ----
mlflow.set_experiment("g1-classificacao-resolution")
with mlflow.start_run(run_name="EXP-matriz-consolidada-sprint06"):
    mlflow.set_tags({"owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
                      "run_type": "consolidated_matrix", "environment": ENVIRONMENT})
    mlflow.log_artifact(str(matriz_path))
    mlflow.log_artifact(str(problematicos_path))
    mlflow.log_artifact(str(chart_path))
    mlflow.log_artifact(str(cobertura_path))
    for k, v in cobertura.items():
        if isinstance(v, (int, float)):
            mlflow.log_metric(k, v)

print("\nTudo registrado no MLflow (run 'EXP-matriz-consolidada-sprint06').")
print(f"Matriz: {matriz_path}")
print(f"Runs problemáticos: {problematicos_path}")
print(f"Gráfico: {chart_path}")
print(f"Cobertura: {cobertura_path}")

## **Medir o efeito das resoluções 224, 384 e 512 no desempenho e custo das CNNs.**

In [ ]:
# ==============================================================================
# EXP-02 / EXP-03 — Ablação de RESOLUÇÃO (224 / 384 / 512) + análise de lesões
# pequenas no CBIS-DDSM.
#
# COMO USAR: este arquivo tem 8 blocos, cada um marcado "### CÉLULA R<n> ###".
# Copie cada bloco (sem o marcador) para UMA célula de código nova no seu
# notebook do Kaggle, na ordem R1 -> R8, sempre DEPOIS da célula
# "Modelo e lógica de execução de um run" (a que define execute_run,
# build_model, CBISDDSMPatchDataset) e DEPOIS da célula de conexão ao MLflow.
# Pode colar como um bloco novo logo após a seção "Run completo" do EXP-01
# original — não precisa mexer em nada que já existe.
#
# Pré-requisito: CONFIG, dicom_lookup, client, mlflow, execute_run,
# build_model, CBISDDSMPatchDataset, build_labeled_dataframe,
# filter_resolvable_rows, resolve_image_path, compute_metrics,
# full_metrics_with_ci, sensitivity_at_specificity, WORKDIR, ENVIRONMENT
# já precisam existir na sessão (são definidos nas células originais do
# baselines-kaggle.ipynb, ANTES do smoke test / run completo do EXP-01).
# ==============================================================================
 
 
# ### CÉLULA R1 ###
# ------------------------------------------------------------------------
# Garante que o MlflowClient existe nesta sessão, independente de qual
# célula do notebook original você rodou antes (a célula que originalmente
# definia `client` fica logo após o "Run completo" do EXP-01 — se você
# pulou essa célula por estar reaproveitando o EXP-01 já existente, esta
# linha evita o NameError: name 'client' is not defined).
# ------------------------------------------------------------------------
 
client = mlflow.tracking.MlflowClient()
 
# Configs de resolução — CADA UMA é cópia EXATA do CONFIG do EXP-01
# (baseline), mudando APENAS image_size e experiment_id. Split, seed,
# arquitetura, batch_size, learning_rate, weight_decay e épocas ficam
# CONGELADOS, conforme o critério de aceite desta task.
# ------------------------------------------------------------------------
 
CONFIG_RES_224 = {**CONFIG, "experiment_id": "EXP-01", "image_size": 224, "ablation_axis": "resolution"}
CONFIG_RES_384 = {**CONFIG, "experiment_id": "EXP-02", "image_size": 384, "ablation_axis": "resolution"}
CONFIG_RES_512 = {**CONFIG, "experiment_id": "EXP-03", "image_size": 512, "ablation_axis": "resolution"}
 
RESOLUTION_CONFIGS = [CONFIG_RES_224, CONFIG_RES_384, CONFIG_RES_512]
 
# Conferência automática: só image_size (e experiment_id) pode diferir entre
# os 3 configs. Se qualquer outro campo divergir, o script para com erro
# em vez de deixar rodar uma ablação inválida.
campos_para_conferir = ["seed", "architecture", "pretrained_weights", "batch_size",
                         "learning_rate", "weight_decay", "epochs", "dataset_root",
                         "train_csvs", "test_csvs", "val_fraction"]
for campo in campos_para_conferir:
    valores = {c["experiment_id"]: c[campo] for c in RESOLUTION_CONFIGS}
    valores_str = {k: str(v) for k, v in valores.items()}  # listas não são hashable
    if len(set(valores_str.values())) > 1:
        raise RuntimeError(f"Campo '{campo}' difere entre os configs: {valores} "
                            f"— isso quebra o critério de 'só a resolução muda'.")
print("Conferência OK: os 3 configs são idênticos, exceto image_size.")
for c in RESOLUTION_CONFIGS:
    print(f"  {c['experiment_id']}: image_size={c['image_size']}")

In [ ]:
# ### CÉLULA R2 ###
# ------------------------------------------------------------------------
# Probe de VRAM — roda 1 passo de treino (forward+backward) em batch_size
# CHEIO (32) nas resoluções 384 e 512 ANTES de comprometer o run completo
# de 20 épocas. Isso importa porque o smoke_test do execute_run usa só
# 16 imagens de treino, e como batch_size real (32) > 16, o smoke test
# acaba rodando com batch efetivo menor (min(batch_size, len(dataset))) —
# ele PODE passar mesmo que o run completo (batch 32 de verdade) estoure
# a VRAM da GPU do Kaggle (Tesla T4, 16GB). Este probe fecha essa lacuna.
# ------------------------------------------------------------------------
 
def vram_probe(image_size, batch_size, architecture="resnet50"):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        print(f"[probe] Sem GPU disponível — pulando probe para image_size={image_size}.")
        return None
 
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
 
    model = build_model(architecture, pretrained=True).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
 
    dummy_x = torch.randn(batch_size, 3, image_size, image_size, device=device)
    dummy_y = torch.randint(0, 2, (batch_size, 1), device=device).float()
 
    optimizer.zero_grad()
    logits = model(dummy_x)
    loss = criterion(logits, dummy_y)
    loss.backward()
    optimizer.step()
 
    peak_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    print(f"[probe] image_size={image_size} | batch_size={batch_size} -> pico de VRAM: {peak_mb:.0f} MB")
 
    del model, optimizer, dummy_x, dummy_y, logits, loss
    torch.cuda.empty_cache()
    return peak_mb
 
 
print("=== PROBE DE VRAM (batch_size cheio, 1 passo) ===")
probe_results = {}
for cfg in [CONFIG_RES_384, CONFIG_RES_512]:
    probe_results[cfg["image_size"]] = vram_probe(cfg["image_size"], CONFIG["batch_size"])
 
for image_size, peak_mb in probe_results.items():
    if peak_mb is not None and peak_mb > 13000:
        print(f"[AVISO] image_size={image_size}: pico de {peak_mb:.0f}MB está perto do limite da "
              f"T4 (16GB). Risco real de CUDA OOM no run completo de 20 épocas (o pico tende a "
              f"subir um pouco com otimizador/ativações acumuladas). Considere rodar essa "
              f"resolução sozinha (sem outra célula competindo por VRAM) e monitore a saída.")

In [ ]:
# ### CÉLULA R3 ###
# ------------------------------------------------------------------------
# Smoke test obrigatório para 384 e 512 — mesmo padrão do EXP-01 original.
# Roda ANTES do run completo de cada resolução nova.
# ------------------------------------------------------------------------
 
EXPERIMENT_NAME_SMOKE = "g1-classificacao-smoke-test"  # já existe no MLflow
 
for cfg in [CONFIG_RES_384, CONFIG_RES_512]:
    print(f"\n=== SMOKE TEST — {cfg['experiment_id']} (image_size={cfg['image_size']}) ===")
    smoke_run_id = execute_run(cfg, smoke_test=True, mlflow_experiment=EXPERIMENT_NAME_SMOKE,
                                max_smoke_samples=16)
    print(f"Smoke test OK. run_id={smoke_run_id}")
 
print("\nSmoke tests concluídos. Só prossiga para a CÉLULA R4 se AMBOS passaram sem erro.")

In [ ]:
# ### CÉLULA R3.5 ###
# ------------------------------------------------------------------------
# Infra de mixed precision (AMP/fp16) — USADA SÓ PARA 512px, porque o probe
# da CÉLULA R2 mostrou 13951MB de pico (~87% da T4) em fp32, risco real de
# CUDA OOM ao longo de 20 épocas por fragmentação de memória acumulada.
#
# IMPORTANTE — isto NÃO é uma mudança de hiperparâmetro do experimento:
# arquitetura, batch_size, learning_rate, weight_decay, épocas, seed e
# split continuam EXATAMENTE os mesmos. É só o formato numérico usado
# internamente no forward/backward (fp16 autocast + GradScaler em vez de
# fp32 puro) — uma adaptação de engenharia pra caber na GPU disponível.
# Fica marcado explicitamente no MLflow (tag `technical_adaptation` +
# param `precision`) e deve ser citado como ressalva no relatório: a
# comparação de VRAM entre 512px e as outras resoluções não é 100%
# apples-to-apples, já que só 512px roda em fp16.
#
# 224px e 384px NÃO usam este caminho — ficam no execute_run original
# (fp32), sem necessidade de AMP (8858MB em 384px é folga confortável).
# ------------------------------------------------------------------------
 
from torch.cuda.amp import GradScaler
 
 
def run_epoch_amp(model, loader, criterion, optimizer, scaler, device, train: bool):
    model.train() if train else model.eval()
    total_loss, all_labels, all_scores = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels, _pid, _path in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True).unsqueeze(1)
            if train:
                optimizer.zero_grad()
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                logits = model(images)
                loss = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * images.size(0)
            all_scores.extend(torch.sigmoid(logits.float()).detach().cpu().numpy().flatten().tolist())
            all_labels.extend(labels.detach().cpu().numpy().flatten().tolist())
    metrics = compute_metrics(all_labels, all_scores)
    metrics["loss"] = total_loss / len(loader.dataset)
    return metrics
 
 
def execute_run_amp(config: dict, smoke_test: bool, mlflow_experiment: str,
                     max_smoke_samples: int = 16, extra_tags: dict = None):
    """Cópia de execute_run com o loop de treino/validação em mixed precision
    (fp16 autocast + GradScaler). Todo o resto — logging MLflow, avaliação
    via evaluate.py compartilhado, checkpoint, registro de falhas — é
    idêntico ao execute_run original, para manter rastreabilidade igual."""
    set_seed(config["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_cuda = device.type == "cuda"
    scaler = GradScaler(enabled=use_cuda)
 
    mlflow.set_experiment(mlflow_experiment)
 
    with mlflow.start_run(run_name=f"{config['experiment_id']}{'-smoke' if smoke_test else ''}") as run:
        mlflow.set_tags({
            "owner": config["owner"], "role": config["role"],
            "status": "exploratory" if smoke_test else "success",
            "ablation_axis": "none" if smoke_test else config["ablation_axis"],
            "environment": ENVIRONMENT,
            "technical_adaptation": "amp_fp16_due_to_vram",
            **(extra_tags or {}),
        })
 
        mlflow.log_params({
            "code_version": CODE_VERSION, "seed": config["seed"],
            "dataset_manifest": ",".join(config["train_csvs"] + config["test_csvs"]),
            "split_strategy": "CBIS-DDSM_official_train_test + patient-level_val_split",
            "architecture": config["architecture"], "pretrained_weights": config["pretrained_weights"],
            "image_size": config["image_size"], "batch_size": config["batch_size"],
            "learning_rate": config["learning_rate"], "weight_decay": config["weight_decay"],
            "epochs": 1 if smoke_test else config["epochs"],
            "preprocessing_version": config["preprocessing_version"], "environment": ENVIRONMENT,
            "precision": "amp_fp16",
        })
 
        try:
            dicom_lookup_local = build_dicom_info_lookup(config["dataset_root"])
            train_full_df = build_labeled_dataframe(
                config["dataset_root"], config["train_csvs"], config["malignant_labels"],
                config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
            )
            train_full_df = filter_resolvable_rows(train_full_df, config["dataset_root"],
                                                     config["image_path_column"], dicom_lookup_local)
            train_df, val_df = patient_level_split(train_full_df, config["val_fraction"], config["seed"])
 
            if smoke_test:
                train_df = train_df.head(max_smoke_samples)
                val_df = val_df.head(max(4, max_smoke_samples // 4))
 
            mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
            train_ds = CBISDDSMPatchDataset(train_df, config["dataset_root"], config["image_path_column"],
                                             config["image_size"], mean, std, augment=not smoke_test,
                                             dicom_lookup=dicom_lookup_local)
            val_ds = CBISDDSMPatchDataset(val_df, config["dataset_root"], config["image_path_column"],
                                           config["image_size"], mean, std, augment=False,
                                           dicom_lookup=dicom_lookup_local)
 
            NUM_WORKERS = 2
            train_loader = DataLoader(train_ds, batch_size=min(config["batch_size"], len(train_ds)),
                                       shuffle=True, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                       persistent_workers=NUM_WORKERS > 0)
            val_loader = DataLoader(val_ds, batch_size=min(config["batch_size"], len(val_ds)),
                                     shuffle=False, num_workers=NUM_WORKERS, pin_memory=use_cuda,
                                     persistent_workers=NUM_WORKERS > 0)
 
            model = build_model(config["architecture"], config["pretrained_weights"] == "imagenet").to(device)
            criterion = nn.BCEWithLogitsLoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"],
                                          weight_decay=config["weight_decay"])
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
 
            n_epochs = 1 if smoke_test else config["epochs"]
            history = []
            best_val_auroc, checkpoint_path = 0.0, os.path.join(WORKDIR, f"{run.info.run_id}_best.pt")
 
            for epoch in range(1, n_epochs + 1):
                t0 = time.time()
                if use_cuda:
                    torch.cuda.reset_peak_memory_stats(device)
 
                train_m = run_epoch_amp(model, train_loader, criterion, optimizer, scaler, device, train=True)
                val_m = run_epoch_amp(model, val_loader, criterion, optimizer, scaler, device, train=False)
                scheduler.step(val_m["auroc"])
                dt = time.time() - t0
 
                peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if use_cuda else None
 
                mlflow.log_metric("train_loss", train_m["loss"], step=epoch)
                mlflow.log_metric("val_loss", val_m["loss"], step=epoch)
                mlflow.log_metric("train_auroc", train_m["auroc"], step=epoch)
                mlflow.log_metric("val_auroc", val_m["auroc"], step=epoch)
                if peak_vram_mb is not None:
                    mlflow.log_metric("peak_vram_mb_epoch", peak_vram_mb, step=epoch)
 
                history.append({"epoch": epoch, "train_loss": train_m["loss"], "val_loss": val_m["loss"],
                                 "train_auroc": train_m["auroc"], "val_auroc": val_m["auroc"],
                                 "seconds": dt, "peak_vram_mb": peak_vram_mb})
 
                vram_str = f"{peak_vram_mb:.0f}MB" if peak_vram_mb is not None else "N/A (CPU)"
                print(f"[AMP] Época {epoch}/{n_epochs} | val_auroc={val_m['auroc']:.4f} | "
                      f"{dt:.1f}s | VRAM pico: {vram_str}")
 
                if val_m["auroc"] > best_val_auroc:
                    best_val_auroc = val_m["auroc"]
                    torch.save({"model_state_dict": model.state_dict(), "architecture": config["architecture"],
                                "image_size": config["image_size"], "seed": config["seed"],
                                "experiment_id": config["experiment_id"]}, checkpoint_path)
 
            duration_seconds = sum(h["seconds"] for h in history)
            avg_epoch_seconds = duration_seconds / len(history) if history else None
            vram_values = [h["peak_vram_mb"] for h in history if h["peak_vram_mb"] is not None]
 
            mlflow.log_metric("duration_seconds", duration_seconds)
            if avg_epoch_seconds is not None:
                mlflow.log_metric("avg_epoch_time_seconds", avg_epoch_seconds)
            if vram_values:
                mlflow.log_metric("peak_vram_mb_max", max(vram_values))
            if use_cuda:
                mlflow.log_param("gpu_name", torch.cuda.get_device_name(device))
 
            history_path = os.path.join(WORKDIR, f"{run.info.run_id}_history.csv")
            pd.DataFrame(history).to_csv(history_path, index=False)
            mlflow.log_artifact(history_path)
            os.remove(history_path)
 
            config_path = os.path.join(WORKDIR, f"{run.info.run_id}_config.json")
            with open(config_path, "w", encoding="utf-8") as f:
                json.dump(config, f, indent=2, ensure_ascii=False)
            mlflow.log_artifact(config_path)
            os.remove(config_path)
 
            if not smoke_test:
                test_df = build_labeled_dataframe(
                    config["dataset_root"], config["test_csvs"], config["malignant_labels"],
                    config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
                )
                test_df = filter_resolvable_rows(test_df, config["dataset_root"],
                                                  config["image_path_column"], dicom_lookup_local)
                test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                                config["image_size"], mean, std, augment=False,
                                                dicom_lookup=dicom_lookup_local)
                test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False,
                                          num_workers=NUM_WORKERS, pin_memory=use_cuda)
 
                model.eval()
                records = []
                with torch.no_grad():
                    for images, labels, patient_ids, _paths in test_loader:
                        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_cuda):
                            probs = torch.sigmoid(model(images.to(device))).float().cpu().numpy().flatten()
                        for pid, lab, prob in zip(patient_ids, labels.numpy().flatten(), probs):
                            records.append({"patient_id": pid, "true_label": int(lab),
                                             "predicted_probability": float(prob)})
 
                predictions_df = pd.DataFrame(records)
 
                print("[AMP] Rodando evaluate.py compartilhado...")
                shared_metrics = run_shared_evaluate(predictions_df, run.info.run_id)
 
                if shared_metrics is not None:
                    log_metrics_dict_to_mlflow(shared_metrics)
                    mlflow.set_tag("evaluation_source", "evaluate_shared")
                else:
                    metrics_test = full_metrics_with_ci(
                        predictions_df["true_label"].values, predictions_df["predicted_probability"].values,
                        n_bootstrap=1000, seed=config["seed"],
                    )
                    log_metrics_dict_to_mlflow(metrics_test)
                    mlflow.set_tag("evaluation_source", "internal_fallback")
                    print("[AVISO] evaluate.py compartilhado falhou — usadas métricas internas como fallback.")
 
                if os.path.exists(checkpoint_path):
                    mlflow.log_artifact(checkpoint_path)
                    os.remove(checkpoint_path)
            else:
                if os.path.exists(checkpoint_path):
                    os.remove(checkpoint_path)
 
            mlflow.set_tag("status", "success")
            return run.info.run_id
 
        except Exception as e:
            mlflow.set_tag("status", "failed")
            mlflow.log_param("error_message", str(e)[:250])
            print(f"[ERRO][AMP] Run falhou, mas foi registrado no MLflow com status=failed: {e}")
            raise
 
 
# Smoke test específico da variante AMP para 512px — roda ANTES do run
# completo, mesmo padrão da CÉLULA R3, só que usando execute_run_amp.
print("=== SMOKE TEST (AMP) — EXP-03 (image_size=512) ===")
smoke_amp_id = execute_run_amp(CONFIG_RES_512, smoke_test=True,
                                mlflow_experiment=EXPERIMENT_NAME_SMOKE, max_smoke_samples=16)
print(f"Smoke test (AMP) OK. run_id={smoke_amp_id}")
print("Só prossiga para a CÉLULA R4 se este smoke test passou sem erro.")

In [ ]:
# ### CÉLULA R4 ###
# ------------------------------------------------------------------------
# Run completo de EXP-02 (384px, fp32 normal) e EXP-03 (512px, AMP/fp16
# pelo motivo explicado na CÉLULA R3.5). O EXP-01 (224px) NÃO é re-rodado —
# reaproveita o run já validado no MLflow (mesma config, mesma seed, mesmo
# split), o que também economiza orçamento de GPU.
#
# Cada resolução roda dentro do seu próprio try/except: se uma falhar,
# a outra continua e o run falho fica registrado no MLflow com
# status=failed, em vez de travar a ablação inteira.
# ------------------------------------------------------------------------
 
EXPERIMENT_NAME_RESOLUTION = "g1-classificacao-resolution"
resolution_run_ids = {}
 
client = mlflow.tracking.MlflowClient()  # reforço: idempotente, sem custo/efeito colateral
exp_res = client.get_experiment_by_name(EXPERIMENT_NAME_RESOLUTION)
exp01_candidatos = [
    r for r in client.search_runs([exp_res.experiment_id], order_by=["start_time ASC"])
    if r.data.tags.get("mlflow.runName") == "EXP-01" and r.data.tags.get("status") == "success"
]
if exp01_candidatos:
    exp01_run = exp01_candidatos[-1]  # o mais recente já validado
    resolution_run_ids["EXP-01"] = {"run_id": exp01_run.info.run_id, "image_size": 224, "status": "success",
                                     "precision": "fp32"}
    print(f"Reaproveitando EXP-01 (224px) já existente no MLflow: run_id={exp01_run.info.run_id}")
else:
    print("[AVISO] Nenhum EXP-01 (224px) com status=success encontrado no MLflow — "
          "rode o baseline original antes de continuar, ou a tabela final ficará incompleta.")
    # run_id_224 = execute_run(CONFIG_RES_224, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_RESOLUTION)
    # resolution_run_ids["EXP-01"] = {"run_id": run_id_224, "image_size": 224, "status": "success", "precision": "fp32"}

# 384px -> execute_run normal (fp32). 512px -> execute_run_amp (fp16), pelo
# risco de OOM identificado no probe da CÉLULA R2.
plano_execucao = [
    (CONFIG_RES_384, execute_run, "fp32"),
    (CONFIG_RES_512, execute_run_amp, "amp_fp16"),
]
 
for cfg, runner, precision in plano_execucao:
    print(f"\n{'='*60}")
    print(f"Rodando {cfg['experiment_id']} (image_size={cfg['image_size']}, precision={precision})")
    print(f"{'='*60}")
    try:
        run_id = runner(cfg, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_RESOLUTION)
        resolution_run_ids[cfg["experiment_id"]] = {"run_id": run_id, "image_size": cfg["image_size"],
                                                      "status": "success", "precision": precision}
        print(f"OK — {cfg['experiment_id']} concluído. run_id: {run_id}")
    except Exception as e:
        resolution_run_ids[cfg["experiment_id"]] = {"run_id": None, "image_size": cfg["image_size"],
                                                      "status": "failed", "error": str(e)[:200],
                                                      "precision": precision}
        print(f"FALHOU — {cfg['experiment_id']} (image_size={cfg['image_size']}): {e}")
        print("Registrado no MLflow com status=failed. Seguindo para a próxima resolução.")
 
print("\n=== RESUMO ===")
for exp_id, info in resolution_run_ids.items():
    print(f"  {exp_id} (image_size={info['image_size']}, precision={info.get('precision')}): "
          f"{info['status']} | run_id: {info['run_id']}")

In [ ]:
# ### CÉLULA R5 ###
# ------------------------------------------------------------------------
# Entrega 1: tabela por resolução (AUROC, AUPRC, sensibilidade a
# especificidade fixa de 90%, VRAM de pico, tempo médio por época).
# ------------------------------------------------------------------------
 
linhas_tabela = []
for exp_id, info in resolution_run_ids.items():
    row = {"experiment_id": exp_id, "image_size": info["image_size"], "status": info["status"],
           "run_id": info["run_id"] or "N/A"}
    if info["run_id"]:
        run = client.get_run(info["run_id"])
        m = run.data.metrics
        p = run.data.params
        row.update({
            "precision": p.get("precision", "fp32"),
            "auroc": m.get("auroc"),
            "auprc": m.get("auprc"),
            "sensitivity_at_spec90": m.get("sensitivity_at_90_specificity",
                                            m.get("sensitivity_at_specificity_90")),
            "avg_epoch_time_seconds": m.get("avg_epoch_time_seconds"),
            "peak_vram_mb_max": m.get("peak_vram_mb_max"),
        })
    linhas_tabela.append(row)
 
tabela_resolucao = pd.DataFrame(linhas_tabela).sort_values("image_size")
tabela_path = os.path.join(WORKDIR, "tabela_resolucao.csv")
tabela_resolucao.to_csv(tabela_path, index=False)
 
print("=== TABELA POR RESOLUÇÃO ===")
print(tabela_resolucao.to_string(index=False))
print(f"\nSalva em: {tabela_path}")
 
if "amp_fp16" in tabela_resolucao["precision"].values:
    print("\n[RESSALVA PARA O RELATÓRIO] O run de 512px usou mixed precision (fp16) por "
          "necessidade de VRAM (probe indicou 13951MB / ~87% da T4 em fp32). A coluna "
          "'peak_vram_mb_max' portanto NÃO é diretamente comparável entre 512px e as demais "
          "resoluções (fp16 reduz o pico de memória por si só, independente do custo real de "
          "computação da resolução maior). AUROC/AUPRC/sensibilidade seguem comparáveis "
          "normalmente — fp16 com GradScaler não muda a arquitetura, os dados nem os "
          "hiperparâmetros, só a precisão numérica interna do forward/backward.")

In [ ]:
dicom_lookup = build_dicom_info_lookup(CONFIG["dataset_root"])

In [ ]:
 ### CÉLULA R6 ###
# ------------------------------------------------------------------------
# Entrega 2 (parte 1): monta a tabela de tamanho de lesão a partir da
# máscara ROI oficial do CBIS-DDSM ("ROI mask file path"), usando a MESMA
# tradução .dcm -> .jpg (via dicom_info.csv) já usada para as imagens
# recortadas. O tamanho da lesão é uma propriedade do CASO clínico, não
# da resolução de treino do CNN — por isso é calculado UMA vez só e depois
# reutilizado para as 3 resoluções.
#
# LIMITAÇÃO IMPORTANTE (documentar no relatório): a área é medida em
# PIXELS da máscara na resolução nativa do scan digitalizado — o pipeline
# não tem acesso a pixel spacing/DPI do CBIS-DDSM, então não dá pra
# converter para mm² com confiança. "Pequena" aqui = tercil inferior de
# área dentro do PRÓPRIO conjunto de teste, não um limiar clínico
# absoluto (tipo "<10mm"). Se seu grupo tiver acesso ao pixel spacing dos
# DICOMs originais, vale substituir esse tercil por um corte em mm reais.
# ------------------------------------------------------------------------
 
ROI_MASK_COLUMN = "ROI mask file path"  # nome oficial da coluna no CBIS-DDSM
 
if "dicom_lookup" not in globals():
    dicom_lookup = build_dicom_info_lookup(CONFIG["dataset_root"])
    print(f"dicom_lookup global criado: {len(dicom_lookup)} imagens mapeadas.")
 
 
def build_lesion_size_table(config: dict, dicom_lookup: dict) -> pd.DataFrame:
    test_df = build_labeled_dataframe(
        config["dataset_root"], config["test_csvs"], config["malignant_labels"],
        config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
    )
 
    if ROI_MASK_COLUMN not in test_df.columns:
        raise KeyError(
            f"Coluna de máscara ROI '{ROI_MASK_COLUMN}' não encontrada no CSV de teste. "
            f"Colunas disponíveis: {list(test_df.columns)}. Ajuste ROI_MASK_COLUMN para o "
            f"nome correto e rode esta célula de novo."
        )
 
    test_df = filter_resolvable_rows(test_df, config["dataset_root"], config["image_path_column"], dicom_lookup)
 
    areas, bbox_areas, resolved_img_paths, mask_ok = [], [], [], []
    for _, row in test_df.iterrows():
        img_path = resolve_image_path(config["dataset_root"], row[config["image_path_column"]], dicom_lookup)
        mask_path = resolve_image_path(config["dataset_root"], row[ROI_MASK_COLUMN], dicom_lookup)
        resolved_img_paths.append(img_path)
 
        if not os.path.exists(mask_path):
            mask_ok.append(False)
            areas.append(np.nan)
            bbox_areas.append(np.nan)
            continue
        try:
            mask_arr = np.array(Image.open(mask_path).convert("L"))
            nonzero = mask_arr > 10  # tira ruído de compressão JPEG do fundo preto
            area_px = int(nonzero.sum())
            if area_px == 0:
                raise ValueError("máscara vazia após threshold")
            ys, xs = np.where(nonzero)
            bbox_area = int((ys.max() - ys.min() + 1) * (xs.max() - xs.min() + 1))
            areas.append(area_px)
            bbox_areas.append(bbox_area)
            mask_ok.append(True)
        except Exception:
            mask_ok.append(False)
            areas.append(np.nan)
            bbox_areas.append(np.nan)
 
    test_df = test_df.copy()
    test_df["_resolved_image_path"] = resolved_img_paths
    test_df["_mask_ok"] = mask_ok
    test_df["lesion_area_px"] = areas
    test_df["lesion_bbox_area_px"] = bbox_areas
 
    n_total, n_ok = len(test_df), int(sum(mask_ok))
    print(f"[lesion-size] Máscara ROI resolvida para {n_ok}/{n_total} linhas do teste "
          f"({100 * n_ok / max(n_total, 1):.1f}%).")
    if n_ok < n_total:
        print(f"[lesion-size] {n_total - n_ok} linhas sem máscara válida ficam de fora da "
              f"análise de tamanho (continuam normalmente nas métricas gerais da CÉLULA R5).")
 
    valid = test_df[test_df["_mask_ok"]].copy()
    valid["size_bucket"] = pd.qcut(valid["lesion_area_px"], q=3, labels=["pequena", "media", "grande"])
 
    print("\n[lesion-size] Distribuição por tercil de área (pixels da máscara, dentro do teste):")
    print(valid.groupby("size_bucket", observed=True)["lesion_area_px"].agg(["count", "min", "median", "max"]))
 
    return valid[["_resolved_image_path", "patient_id", "lesion_area_px", "lesion_bbox_area_px", "size_bucket"]]
 
 
lesion_size_table = build_lesion_size_table(CONFIG_RES_224, dicom_lookup)

## ****Esta 6.5 depois ou seja se der algum erro ou sla precisar remover, este foi o último que mexi e até colocar ele estava funcionando tudo****

In [ ]:
# ### CÉLULA R6.5 ###
# ------------------------------------------------------------------------
# Por que este bloco existe: o mlflow.db restaurado traz só os METADADOS
# (params/métricas) do EXP-01 antigo. Os arquivos de artefato — inclusive o
# checkpoint _best.pt — ficavam em /kaggle/working/mlruns da sessão em que
# ele rodou e NÃO vieram junto. Sem checkpoint não dá pra reinferir o teste
# em 224px e estratificar por tamanho de lesão (R7).
#
# Solução: re-treinar 224px UMA vez nesta sessão, com a MESMA config e seed
# (execute_run original, fp32), guardando o checkpoint. O EXP-01 histórico
# é mantido só como referência para checar reprodutibilidade.
# Pule este bloco se o EXP-01 desta sessão já tiver checkpoint.
# ------------------------------------------------------------------------
 
def run_has_checkpoint(run_id):
    return any(a.path.endswith("_best.pt") for a in client.list_artifacts(run_id))
 
if not run_has_checkpoint(resolution_run_ids["EXP-01"]["run_id"]):
    historico_exp01 = dict(resolution_run_ids["EXP-01"])
    hist_metrics = client.get_run(historico_exp01["run_id"]).data.metrics
    print("EXP-01 histórico sem checkpoint — re-treinando 224px (mesma config/seed)...")
 
    cfg_224 = {**CONFIG_RES_224, "experiment_id": "EXP-01-ckpt"}
    run_id_224 = execute_run(cfg_224, smoke_test=False, mlflow_experiment=EXPERIMENT_NAME_RESOLUTION)
    resolution_run_ids["EXP-01"] = {"run_id": run_id_224, "image_size": 224,
                                     "status": "success", "precision": "fp32"}
 
    novo = client.get_run(run_id_224).data.metrics
    print("\n=== Checagem de reprodutibilidade (224px) ===")
    for k in ["auroc", "auprc"]:
        print(f"  {k}: histórico={hist_metrics.get(k)} | re-treino={novo.get(k)}")
    print("Se os valores forem iguais/muito próximos, o re-treino é equivalente ao EXP-01 original.")
else:
    print("EXP-01 desta sessão já tem checkpoint — nada a fazer.")

In [ ]:
# ### CÉLULA R7 ###
# ------------------------------------------------------------------------
# Entrega 2 (parte 2): reinfere o teste com o MELHOR checkpoint de cada
# resolução (baixado de volta do MLflow, já que a política do grupo apaga
# o .pt local após o upload), junta com o tamanho de lesão da CÉLULA R6, e
# calcula AUROC/AUPRC/sensibilidade@especificidade90 POR BUCKET de
# tamanho, POR resolução.
# ------------------------------------------------------------------------
 
def load_model_from_run(run_id: str, architecture: str, device):
    artifacts = client.list_artifacts(run_id)
    ckpt_artifact = next((a for a in artifacts if a.path.endswith("_best.pt")), None)
    if ckpt_artifact is None:
        raise FileNotFoundError(f"Nenhum checkpoint '_best.pt' nos artefatos do run {run_id}.")
    local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=ckpt_artifact.path)
    checkpoint = torch.load(local_path, map_location=device)
    model = build_model(architecture, pretrained=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device).eval()
    return model
 
 
def predict_test_set(config: dict, model, dicom_lookup: dict, device):
    test_df = build_labeled_dataframe(
        config["dataset_root"], config["test_csvs"], config["malignant_labels"],
        config["non_malignant_labels"], config["patient_id_column"], config["pathology_column"],
    )
    test_df = filter_resolvable_rows(test_df, config["dataset_root"], config["image_path_column"], dicom_lookup)
 
    mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
    test_ds = CBISDDSMPatchDataset(test_df, config["dataset_root"], config["image_path_column"],
                                    config["image_size"], mean, std, augment=False, dicom_lookup=dicom_lookup)
    test_loader = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, num_workers=2)
 
    records = []
    with torch.no_grad():
        for images, labels, patient_ids, paths in test_loader:
            probs = torch.sigmoid(model(images.to(device))).cpu().numpy().flatten()
            for pid, lab, prob, path in zip(patient_ids, labels.numpy().flatten(), probs, paths):
                records.append({"patient_id": pid, "true_label": int(lab),
                                 "predicted_probability": float(prob), "_resolved_image_path": path})
    return pd.DataFrame(records)
 
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lesion_rows = []
 
for exp_id, info in resolution_run_ids.items():
    if info["status"] != "success":
        print(f"[skip] {exp_id} não teve run com sucesso — fora da análise de lesões.")
        continue
 
    image_size = info["image_size"]
    cfg = next(c for c in RESOLUTION_CONFIGS if c["image_size"] == image_size)
 
    print(f"\n=== Reinferindo teste para {exp_id} (image_size={image_size}) ===")
    model = load_model_from_run(info["run_id"], cfg["architecture"], device)
    preds_df = predict_test_set(cfg, model, dicom_lookup, device)
    merged = preds_df.merge(lesion_size_table, on=["_resolved_image_path", "patient_id"], how="inner")
    print(f"  {len(merged)} linhas casadas com tamanho de lesão conhecido "
          f"(de {len(preds_df)} previsões totais).")
 
    for bucket in ["pequena", "media", "grande", "TODAS"]:
        subset = merged if bucket == "TODAS" else merged[merged["size_bucket"] == bucket]
        if len(subset) < 10 or subset["true_label"].nunique() < 2:
            lesion_rows.append({"experiment_id": exp_id, "image_size": image_size, "size_bucket": bucket,
                                 "n": len(subset), "auroc": np.nan, "auroc_ci95_low": np.nan,
                                 "auroc_ci95_high": np.nan, "auprc": np.nan,
                                 "sensitivity_at_spec90": np.nan,
                                 "obs": "amostra insuficiente ou só uma classe presente"})
            continue
        m = full_metrics_with_ci(subset["true_label"].values, subset["predicted_probability"].values,
                                  n_bootstrap=1000, seed=cfg["seed"])
        sens90, _ = sensitivity_at_specificity(subset["true_label"].values,
                                                subset["predicted_probability"].values, 0.90)
        lesion_rows.append({"experiment_id": exp_id, "image_size": image_size, "size_bucket": bucket,
                             "n": len(subset), "auroc": m["auroc"],
                             "auroc_ci95_low": m["auroc_ci95"][0], "auroc_ci95_high": m["auroc_ci95"][1],
                             "auprc": m["auprc"], "sensitivity_at_spec90": sens90, "obs": ""})
 
    del model
    torch.cuda.empty_cache()
 
lesion_df = pd.DataFrame(lesion_rows)
lesion_path = os.path.join(WORKDIR, "analise_lesoes_pequenas_por_resolucao.csv")
lesion_df.to_csv(lesion_path, index=False)
 
print("\n=== ANÁLISE DE LESÕES POR TAMANHO E RESOLUÇÃO ===")
print(lesion_df.to_string(index=False))
print(f"\nSalva em: {lesion_path}")

In [ ]:
# ### CÉLULA R8 ###
# ------------------------------------------------------------------------
# Gráfico final (AUROC geral vs. lesões pequenas, por resolução) e registro
# de tudo no MLflow como evidência — mesmo padrão da matriz consolidada.
# ------------------------------------------------------------------------
 
import matplotlib.pyplot as plt
 
fig, ax = plt.subplots(figsize=(8, 5))
for bucket, cor, marcador in [("TODAS", "#1565C0", "o"), ("pequena", "#C62828", "s")]:
    sub = lesion_df[lesion_df["size_bucket"] == bucket].sort_values("image_size")
    ax.plot(sub["image_size"], sub["auroc"], marker=marcador, label=bucket, color=cor)
ax.set(title="AUROC no teste por resolução — geral vs. lesões pequenas",
       xlabel="Resolução de entrada (px)", ylabel="AUROC")
ax.legend(title="Subconjunto do teste")
ax.grid(alpha=0.3)
chart_path = os.path.join(WORKDIR, "auroc_vs_resolucao_lesoes.png")
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()
 
mlflow.set_experiment(EXPERIMENT_NAME_RESOLUTION)
with mlflow.start_run(run_name="EXP-resolucao-analise-lesoes-pequenas"):
    mlflow.set_tags({
        "owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
        "run_type": "lesion_size_analysis", "environment": ENVIRONMENT,
        "test_set_touched": "true", "evaluation_source": "internal_lesion_stratified",
    })
    mlflow.log_artifact(tabela_path)
    mlflow.log_artifact(lesion_path)
    mlflow.log_artifact(chart_path)
 
print("\nTabela por resolução + análise de lesões pequenas registradas no MLflow "
      "(run 'EXP-resolucao-analise-lesoes-pequenas').")
print(f"Tabela: {tabela_path}")
print(f"Análise de lesões: {lesion_path}")
print(f"Gráfico: {chart_path}")

## ****Ablação BATCH × LR — braço "LR fixo" vs braço "regra linear" + interação****

In [ ]:
# ### CÉLULA L1 ###
# ------------------------------------------------------------------------
# Objetivo: braço "LR fixo" (lr=1e-4 constante, batch 8/16/32/64 — já
# existe no MLflow como EXP-13..16) vs braço "regra linear"
# (lr = lr_base * batch/batch_base). batch_size e learning_rate são as
# ÚNICAS variáveis deliberadamente alteradas em cada braço — tudo o mais
# (arquitetura, weight_decay, épocas, seed, split, image_size=224) fica
# idêntico ao CONFIG do EXP-01.
#
# b=32 cai no MESMO ponto nos dois braços (lr=1e-4 é tanto o "fixo" quanto
# o resultado da regra linear em b=batch_base) — por isso aparece com
# arms=["fixed","linear"] e um único run cobre as duas.
# ------------------------------------------------------------------------
 
BASE_BATCH = 32
BASE_LR = CONFIG["learning_rate"]  # 1e-4, igual ao EXP-01
BATCH_SIZES = [8, 16, 32, 64]
 
 
def linear_scaled_lr(batch_size):
    return BASE_LR * (batch_size / BASE_BATCH)
 
 
EXPERIMENT_NAME_BATCH_HIST = "g1-classificacao-batch_size"   # onde já estão EXP-13..16
EXPERIMENT_NAME_BATCH_LR = "g1-classificacao-batch_lr_rule"  # novo, pros runs do braço linear
 
combos = [
    {"batch_size": 8,  "learning_rate": BASE_LR,              "arms": ["fixed"],
     "run_name": "EXP-13",     "experiment": EXPERIMENT_NAME_BATCH_HIST, "reuse": True},
    {"batch_size": 16, "learning_rate": BASE_LR,              "arms": ["fixed"],
     "run_name": "EXP-14",     "experiment": EXPERIMENT_NAME_BATCH_HIST, "reuse": True},
    {"batch_size": 32, "learning_rate": BASE_LR,              "arms": ["fixed", "linear"],
     "run_name": "EXP-15",     "experiment": EXPERIMENT_NAME_BATCH_HIST, "reuse": True},
    {"batch_size": 64, "learning_rate": BASE_LR,              "arms": ["fixed"],
     "run_name": "EXP-16",     "experiment": EXPERIMENT_NAME_BATCH_HIST, "reuse": True},
    {"batch_size": 8,  "learning_rate": linear_scaled_lr(8),  "arms": ["linear"],
     "run_name": "EXP-LR-b08", "experiment": EXPERIMENT_NAME_BATCH_LR,   "reuse": False},
    {"batch_size": 16, "learning_rate": linear_scaled_lr(16), "arms": ["linear"],
     "run_name": "EXP-LR-b16", "experiment": EXPERIMENT_NAME_BATCH_LR,   "reuse": False},
    {"batch_size": 64, "learning_rate": linear_scaled_lr(64), "arms": ["linear"],
     "run_name": "EXP-LR-b64", "experiment": EXPERIMENT_NAME_BATCH_LR,   "reuse": False},
]
 
 
def build_combo_config(combo, experiment_id):
    cfg = {**CONFIG}
    cfg["experiment_id"] = experiment_id
    cfg["batch_size"] = combo["batch_size"]
    cfg["learning_rate"] = combo["learning_rate"]
    cfg["ablation_axis"] = "batch_lr_interaction"
    return cfg
 
 
# Conferência automática: compara cada combo ao CONFIG base, campo a campo,
# ignorando só batch_size/learning_rate/experiment_id/ablation_axis — se
# qualquer OUTRO campo divergir, a ablação não respeita o critério de
# aceite e paramos aqui.
campos_fixos = ["seed", "architecture", "pretrained_weights", "image_size",
                "weight_decay", "epochs", "dataset_root", "train_csvs", "test_csvs", "val_fraction"]
for combo in combos:
    cfg = build_combo_config(combo, combo["run_name"])
    for campo in campos_fixos:
        if str(cfg[campo]) != str(CONFIG[campo]):
            raise RuntimeError(f"Combo {combo['run_name']} difere do CONFIG base em '{campo}' "
                                f"— isso quebra o critério de aceite (só batch/lr podem variar).")
 
print("Conferência OK: em todos os combos, só batch_size e learning_rate mudam.")
print(f"{'run_name':<12} {'braço(s)':<16} {'batch':>6} {'lr':>10} {'reaproveita?':>13}")
for combo in combos:
    print(f"{combo['run_name']:<12} {','.join(combo['arms']):<16} {combo['batch_size']:>6} "
          f"{combo['learning_rate']:>10.2e} {'sim' if combo['reuse'] else 'novo':>13}")

In [ ]:
# ### CÉLULA L2 ###
# ------------------------------------------------------------------------
# Garante client e dicom_lookup nesta sessão, independente do que já foi
# rodado antes (mesmo motivo do NameError visto na task de resolução).
# ------------------------------------------------------------------------
 
client = mlflow.tracking.MlflowClient()
if "dicom_lookup" not in globals():
    dicom_lookup = build_dicom_info_lookup(CONFIG["dataset_root"])
print(f"dicom_lookup pronto: {len(dicom_lookup)} imagens mapeadas.")

In [ ]:
# ### CÉLULA L3 ###
# ------------------------------------------------------------------------
# Resolve run_id dos combos marcados reuse=True, procurando por nome no
# experimento histórico. Nota de VRAM: batch 8/16/32/64 em 224px já rodou
# com sucesso na T4 no histórico (EXP-13..16, pico ~1-3GB, até ~5.7GB em
# batch=64) — não é necessário repetir o probe de VRAM desta vez.
# ------------------------------------------------------------------------
 
def find_successful_run(mlflow_experiment_name, run_name):
    mlflow.set_experiment(mlflow_experiment_name)
    exp = client.get_experiment_by_name(mlflow_experiment_name)
    candidatos = [
        r for r in client.search_runs([exp.experiment_id], order_by=["start_time ASC"])
        if r.data.tags.get("mlflow.runName") == run_name and r.data.tags.get("status") == "success"
    ]
    return candidatos[-1].info.run_id if candidatos else None
 
 
for combo in combos:
    if combo["reuse"]:
        run_id = find_successful_run(combo["experiment"], combo["run_name"])
        combo["run_id"] = run_id
        if run_id:
            print(f"Reaproveitando {combo['run_name']}: run_id={run_id}")
        else:
            print(f"[AVISO] {combo['run_name']} não encontrado com status=success nesta sessão — "
                  f"vai precisar ser rodado do zero no L5. Se achar estranho, confira se o histórico "
                  f"do MLflow foi restaurado corretamente (mesmo bug de nome de arquivo já visto antes).")
            combo["reuse"] = False
    else:
        combo["run_id"] = None

In [ ]:
# ### CÉLULA L4 ###
# ------------------------------------------------------------------------
# Smoke test obrigatório só para os combos que vão rodar de verdade
# (reuse=False): os 3 novos do braço linear, mais qualquer combo
# "reuse" que não foi encontrado no L3.
# ------------------------------------------------------------------------
 
EXPERIMENT_NAME_SMOKE = "g1-classificacao-smoke-test"
 
for combo in combos:
    if combo["run_id"] is not None:
        continue
    cfg = build_combo_config(combo, combo["run_name"])
    print(f"\n=== SMOKE TEST — {combo['run_name']} (batch={combo['batch_size']}, "
          f"lr={combo['learning_rate']:.2e}) ===")
    smoke_id = execute_run(cfg, smoke_test=True, mlflow_experiment=EXPERIMENT_NAME_SMOKE, max_smoke_samples=16)
    print(f"Smoke OK. run_id={smoke_id}")
 
print("\nSmoke tests concluídos. Só prossiga para o L5 se todos passaram sem erro.")

In [ ]:
# ### CÉLULA L5 ###
# ------------------------------------------------------------------------
# Runs completos dos combos que ainda não têm run_id (normalmente só os
# 3 novos do braço linear: b=8/lr=2.5e-5, b=16/lr=5e-5, b=64/lr=2e-4).
# Cada um roda em try/except — se um falhar, os outros continuam.
# ------------------------------------------------------------------------
 
for combo in combos:
    if combo["run_id"] is not None:
        continue
    cfg = build_combo_config(combo, combo["run_name"])
    print(f"\n{'='*60}")
    print(f"Rodando {combo['run_name']} (batch={combo['batch_size']}, lr={combo['learning_rate']:.2e})")
    print(f"{'='*60}")
    try:
        run_id = execute_run(cfg, smoke_test=False, mlflow_experiment=combo["experiment"])
        combo["run_id"] = run_id
        print(f"OK — {combo['run_name']} concluído. run_id: {run_id}")
    except Exception as e:
        combo["run_id"] = None
        combo["error"] = str(e)[:200]
        print(f"FALHOU — {combo['run_name']}: {e}")
        print("Registrado no MLflow com status=failed. Seguindo para o próximo combo.")
 
print("\n=== RESUMO ===")
for combo in combos:
    print(f"  {combo['run_name']} (batch={combo['batch_size']}, lr={combo['learning_rate']:.2e}): "
          f"run_id={combo['run_id']}")

In [ ]:
# ### CÉLULA L6 ###
# ------------------------------------------------------------------------
# Garante checkpoint para TODOS os combos (inclusive os reaproveitados do
# histórico, que podem ter o mesmo problema já visto no EXP-01: artefato
# não veio junto na restauração do mlflow.db). Sem checkpoint não dá pra
# fazer a análise de BatchNorm do L8. Se faltar, re-treina preservando
# batch/lr daquele combo específico.
# ------------------------------------------------------------------------
 
def run_has_checkpoint(run_id):
    return any(a.path.endswith("_best.pt") for a in client.list_artifacts(run_id))
 
 
for combo in combos:
    if combo["run_id"] is None:
        print(f"[skip] {combo['run_name']} sem run_id (falhou no L5) — fora da análise de BatchNorm.")
        continue
    if run_has_checkpoint(combo["run_id"]):
        continue
    print(f"[checkpoint ausente] {combo['run_name']} (batch={combo['batch_size']}, "
          f"lr={combo['learning_rate']:.2e}) — re-treinando nesta sessão para obter checkpoint...")
    cfg = build_combo_config(combo, f"{combo['run_name']}-ckpt")
    new_run_id = execute_run(cfg, smoke_test=False, mlflow_experiment=combo["experiment"])
    combo["run_id"] = new_run_id
    print(f"  OK — novo run_id: {new_run_id}")

In [ ]:
# ### CÉLULA L7 ###
# ------------------------------------------------------------------------
# Entrega 1: tabela com métricas clínicas (AUROC, AUPRC, sensibilidade a
# especificidade fixa de 90%), custo (VRAM de pico, tempo médio/época) e
# estabilidade de treino (desvio-padrão e coeficiente de variação do
# val_auroc nas últimas 5 épocas, a partir do histórico já logado — sem
# precisar de runs extras com seeds diferentes).
# ------------------------------------------------------------------------
 
def training_stability_from_history(run_id, last_n=5):
    try:
        local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=f"{run_id}_history.csv")
        hist = pd.read_csv(local_path)
        tail = hist.tail(last_n)
        std = float(tail["val_auroc"].std())
        mean = float(tail["val_auroc"].mean())
        cv = float(std / mean) if mean else np.nan
        return std, cv
    except Exception as e:
        print(f"  [aviso] não deu pra calcular estabilidade do run {run_id}: {e}")
        return np.nan, np.nan
 
 
linhas = []
for combo in combos:
    for arm in combo["arms"]:
        row = {"arm": arm, "batch_size": combo["batch_size"], "learning_rate": combo["learning_rate"],
               "run_name": combo["run_name"], "run_id": combo["run_id"] or "N/A"}
        if combo["run_id"]:
            run = client.get_run(combo["run_id"])
            m = run.data.metrics
            std, cv = training_stability_from_history(combo["run_id"])
            row.update({
                "auroc": m.get("auroc"),
                "auprc": m.get("auprc"),
                "sensitivity_at_spec90": m.get("sensitivity_at_90_specificity",
                                                m.get("sensitivity_at_specificity_90")),
                "avg_epoch_time_seconds": m.get("avg_epoch_time_seconds"),
                "peak_vram_mb_max": m.get("peak_vram_mb_max"),
                "val_auroc_std_last5": std,
                "val_auroc_cv_last5": cv,
            })
        linhas.append(row)
 
tabela_batch_lr = pd.DataFrame(linhas).sort_values(["arm", "batch_size"])
tabela_path = os.path.join(WORKDIR, "tabela_batch_lr.csv")
tabela_batch_lr.to_csv(tabela_path, index=False)
 
print("=== TABELA BATCH × LR (braço fixo vs regra linear) ===")
print(tabela_batch_lr.to_string(index=False))
print(f"\nSalva em: {tabela_path}")
print("\nNota: val_auroc_cv_last5 mede o quanto o val_auroc oscilou nas últimas 5 épocas — "
      "um proxy de estabilidade de treino sem precisar rodar seeds extras. Quanto menor, mais estável.")

In [ ]:
# ### CÉLULA L8 ###
# ------------------------------------------------------------------------
# Entrega 2: interação com BatchNorm. Carrega o checkpoint de cada combo
# e resume as estatísticas acumuladas (running_mean / running_var) de
# todas as camadas BatchNorm2d. A hipótese clínica/técnica a checar com
# os números reais: batches pequenos (8, 16) calculam running_mean/var em
# cima de poucas amostras por passo -> estimativas mais ruidosas; a regra
# linear ajusta o PASSO do otimizador, não o tamanho do mini-batch que
# alimenta a normalização -- então o ruído de BN tende a aparecer nos dois
# braços para batch pequeno, mas o efeito final no desempenho pode diferir
# conforme o LR também escala ou não.
# ------------------------------------------------------------------------
 
def load_model_from_run(run_id, architecture, device):
    artifacts = client.list_artifacts(run_id)
    ckpt_artifact = next((a for a in artifacts if a.path.endswith("_best.pt")), None)
    if ckpt_artifact is None:
        raise FileNotFoundError(f"Nenhum checkpoint '_best.pt' nos artefatos do run {run_id}.")
    local_path = mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=ckpt_artifact.path)
    checkpoint = torch.load(local_path, map_location=device)
    model = build_model(architecture, pretrained=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device).eval()
    return model
 
 
def bn_stats_summary(model):
    means, varis, n_channels, n_near_zero_var = [], [], 0, 0
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            rm = module.running_mean.detach().cpu().numpy()
            rv = module.running_var.detach().cpu().numpy()
            means.append(np.abs(rm))
            varis.append(rv)
            n_channels += rv.size
            n_near_zero_var += int((rv < 1e-3).sum())
    all_means = np.concatenate(means) if means else np.array([])
    all_vars = np.concatenate(varis) if varis else np.array([])
    return {
        "bn_running_mean_abs_mean": float(all_means.mean()) if all_means.size else np.nan,
        "bn_running_var_mean": float(all_vars.mean()) if all_vars.size else np.nan,
        "bn_running_var_std": float(all_vars.std()) if all_vars.size else np.nan,
        "bn_pct_near_zero_var_channels": (100 * n_near_zero_var / n_channels) if n_channels else np.nan,
        "bn_n_channels": n_channels,
    }
 
 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bn_rows = []
for combo in combos:
    if not combo["run_id"]:
        continue
    try:
        model = load_model_from_run(combo["run_id"], CONFIG["architecture"], device)
    except FileNotFoundError as e:
        print(f"  [skip] {e}")
        continue
    stats = bn_stats_summary(model)
    for arm in combo["arms"]:
        bn_rows.append({"arm": arm, "batch_size": combo["batch_size"],
                         "learning_rate": combo["learning_rate"], "run_name": combo["run_name"], **stats})
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
 
bn_df = pd.DataFrame(bn_rows).sort_values(["arm", "batch_size"])
bn_path = os.path.join(WORKDIR, "analise_batchnorm.csv")
bn_df.to_csv(bn_path, index=False)
 
print("=== ESTATÍSTICAS DE BATCHNORM POR COMBO ===")
print(bn_df.to_string(index=False))
print(f"\nSalva em: {bn_path}")
print("\nComo ler: bn_running_var_mean alto/instável em batches pequenos é o sinal clássico de "
      "estatísticas de BN ruidosas. bn_pct_near_zero_var_channels alto pode indicar canais "
      "quase 'mortos' (colapsados) — compare entre os dois braços no mesmo batch_size para ver "
      "se a regra linear atenua ou não esse efeito.")
 
fig, ax = plt.subplots(figsize=(8, 5))
for arm, cor, marcador in [("fixed", "#1565C0", "o"), ("linear", "#C62828", "s")]:
    sub = bn_df[bn_df["arm"] == arm].sort_values("batch_size")
    ax.plot(sub["batch_size"], sub["bn_running_var_mean"], marker=marcador, label=arm, color=cor)
ax.set(title="Variância média acumulada do BatchNorm por batch_size",
       xlabel="Batch size", ylabel="bn_running_var_mean")
ax.set_xscale("log", base=2)
ax.legend(title="Braço")
ax.grid(alpha=0.3)
chart_path = os.path.join(WORKDIR, "batchnorm_vs_batch_size.png")
fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
# ### CÉLULA L9 ###
# ------------------------------------------------------------------------
# Registro final no MLflow como evidência, mesmo padrão das outras tasks.
# ------------------------------------------------------------------------
 
mlflow.set_experiment(EXPERIMENT_NAME_BATCH_LR)
with mlflow.start_run(run_name="EXP-batchlr-analise-final"):
    mlflow.set_tags({
        "owner": CONFIG["owner"], "role": CONFIG["role"], "status": "success",
        "run_type": "batch_lr_rule_analysis", "environment": ENVIRONMENT,
        "test_set_touched": "true", "evaluation_source": "internal_combined",
    })
    mlflow.log_artifact(tabela_path)
    mlflow.log_artifact(bn_path)
    mlflow.log_artifact(chart_path)
 
print("\nTabela batch×LR + análise de BatchNorm registradas no MLflow "
      "(run 'EXP-batchlr-analise-final').")
print(f"Tabela: {tabela_path}")
print(f"Análise BN: {bn_path}")
print(f"Gráfico: {chart_path}")